In [5]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from time import perf_counter
from functools import partial
from pathlib import Path
import random
from collections import defaultdict
from contextlib import contextmanager
import json
from tqdm.autonotebook import tqdm
from datetime import datetime
from itertools import count
import shutil
import re
from math import sqrt
from typing import Dict, List, Iterable, Optional
from joblib import Parallel, delayed
import joblib
from threadpoolctl import threadpool_limits

from sklearn.base import clone 
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.utils import check_random_state
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import MultiTaskElasticNet
import xgboost as xgb

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from hyperopt.early_stop import no_progress_loss
from hyperopt.pyll import scope

import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use("ggplot")
sns.set_context("paper")

import warnings
warnings.filterwarnings("ignore")

# Set random seed for reproducibility
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

### Load Data

In [6]:
BASE_DIR = Path("../data/features_label")
SUBSETS_DIR = BASE_DIR / "subsets_x"
Y_PATH = BASE_DIR / "tca_status_medium.csv"

In [7]:
X_union        = pd.read_csv(SUBSETS_DIR / "X_subset__union__thr0.70.csv", index_col=0)
X_intersection = pd.read_csv(SUBSETS_DIR / "X_subset__intersection__thr0.70.csv", index_col=0)
X_xgb_only     = pd.read_csv(SUBSETS_DIR / "X_subset__xgb_only__thr0.70.csv", index_col=0)
X_en_only      = pd.read_csv(SUBSETS_DIR / "X_subset__en_only__thr0.70.csv", index_col=0)
X_xgb_all       = pd.read_csv(SUBSETS_DIR / "X_subset__xgb_all__thr0.70.csv", index_col=0)
X_en_all        = pd.read_csv(SUBSETS_DIR / "X_subset__en_all__thr0.70.csv", index_col=0)

print(f"[LOAD] union={X_union.shape}, intersection={X_intersection.shape}, "
      f"xgb_only={X_xgb_only.shape}, en_only={X_en_only.shape}, "
      f"xgb_all={X_xgb_all.shape}, en_all={X_en_all.shape}")

Y = pd.read_csv(Y_PATH, index_col=0)
Y = Y[["TCA non canonical", "activity TCA non canonical"]].astype("float32")
print(f"[Y] shape: {Y.shape}")

[LOAD] union=(513, 436), intersection=(513, 72), xgb_only=(513, 114), en_only=(513, 250), xgb_all=(513, 186), en_all=(513, 322)
[Y] shape: (513, 2)


In [8]:
# Align all datasets to a common sample index
common_idx = (
    X_union.index
    .intersection(X_intersection.index)
    .intersection(X_xgb_only.index)
    .intersection(X_en_only.index)
    .intersection(Y.index)
)

# Filter and reindex
X_union        = X_union.loc[common_idx]
X_intersection = X_intersection.loc[common_idx]
X_xgb_only     = X_xgb_only.loc[common_idx]
X_en_only      = X_en_only.loc[common_idx]
X_xgb_all      = X_xgb_all.loc[common_idx]
X_en_all       = X_en_all.loc[common_idx]
Y              = Y.loc[common_idx]

print(f"[ALIGN] Common samples: {len(common_idx)}")
print(f"  X_union:        {X_union.shape}")
print(f"  X_intersection:{X_intersection.shape}")
print(f"  X_xgb_only:    {X_xgb_only.shape}")
print(f"  X_en_only:     {X_en_only.shape}")
print(f"  X_xgb_all:     {X_xgb_all.shape}")
print(f"  X_en_all:      {X_en_all.shape}")
print(f"  Y:             {Y.shape}")
y_mat = Y.values.astype("float32")

[ALIGN] Common samples: 513
  X_union:        (513, 436)
  X_intersection:(513, 72)
  X_xgb_only:    (513, 114)
  X_en_only:     (513, 250)
  X_xgb_all:     (513, 186)
  X_en_all:      (513, 322)
  Y:             (513, 2)


### ElasticNet Multitask

In [9]:
@contextmanager
def fold_logger(outdir, fold_id: int, also_stdout: bool = True):
    outdir.mkdir(parents=True, exist_ok=True)
    log_path = outdir / f"log_fold_{fold_id}.txt"
    f = open(log_path, "a", buffering=1, encoding="utf-8")
    def log(msg: str):
        ts = datetime.now().strftime("%H:%M:%S")
        line = f"[{ts}] [fold {fold_id}] {msg}"
        if also_stdout:
            print(line, flush=True)
        f.write(line + "\n")
    try:
        yield log, log_path
    finally:
        try:
            f.close()
        except Exception:
            pass

def to_json_compatible(obj):
    """Recursively convert numpy types to standard JSON types."""
    if isinstance(obj, (np.integer, )):
        return int(obj)
    if isinstance(obj, (np.floating, )):
        return float(obj)
    if isinstance(obj, dict):
        return {to_json_compatible(k): to_json_compatible(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_json_compatible(v) for v in obj]
    return obj

#  EN pipeline builder
def make_pipeline_en(seed: int = 42) -> Pipeline:
    """
    Standardize X then fit MultiTaskElasticNet.
    y-scaling is handled outside the pipeline (per-fold) to invert predictions back.
    """
    return Pipeline(steps=[
        ("scale_x", StandardScaler(with_mean=True, with_std=True)),
        ("model", MultiTaskElasticNet(random_state=seed, max_iter=10000, tol=1e-6))
    ])

def nested_cv_en_phase1(
    X_df: pd.DataFrame,
    y_mat: np.ndarray,
    outer_cv: KFold,
    inner_cv: KFold,
    hyperopt_space: dict | None = None,
    hyperopt_evals: int = 200,
    early_stop: int = 50,
    checkpoint_every: int = 10,
    interim_eval_every: int = 20,
    use_one_se: bool = True,
    prefer_stronger_in_1se: bool = True,
    n_jobs_inner: int = -1,
    n_jobs_outer: int = 1,
    outdir: Path | None = None,
    seed: int = 42,
):
    """
    ElasticNet nested CV with Hyperopt and full XGB-like IO/parallelization.
    - Per-fold logs: log_fold_{k}.txt
    - Inner-CV: Hyperopt over (alpha, l1_ratio), objective = mean MSE on z-scaled y
    - Final pick via 1-SE (optional; prefer stronger regularization if requested)
    - Saves per-fold: predictions (train/test), pipeline .joblib, metadata .json, metrics_fold_{k}.csv
    - Saves per-panel: all_metadata.csv, metrics_all_folds.csv, metrics_summary_test.csv
    Returns: (all_meta_df, metrics_all_folds_df, summary_test_df)
    """
    assert outdir is not None, "Please provide an output directory."
    outdir.mkdir(parents=True, exist_ok=True)

    X = X_df.values.astype(np.float32)
    splits = list(outer_cv.split(X))

    # Default Hyperopt space if not provided
    if hyperopt_space is None:
        hyperopt_space = {
            "alpha": hp.choice("alpha_bucket", [
                hp.loguniform("a_mid",  np.log(1e-4), np.log(1.0)),
                hp.loguniform("a_str",  np.log(5e-2), np.log(3.0)),
            ]),
            "l1_ratio": hp.uniform("l1_ratio", 0.2, 0.9),
        }

    def _run_one_fold(fold_id, tr_idx, te_idx):
        # Limit threads inside each parallel job to avoid oversubscription
        with threadpool_limits(limits=1):
            with fold_logger(outdir, fold_id) as (log, log_path):
                t0 = perf_counter()
                log(f"START fold — train={len(tr_idx)}, test={len(te_idx)}, seed={seed + fold_id}")

                X_tr, X_te = X[tr_idx], X[te_idx]
                y_tr, y_te = y_mat[tr_idx], y_mat[te_idx]

                # y-scaling outside pipeline to invert predictions to original scale
                y_scaler = StandardScaler()
                y_tr_z = y_scaler.fit_transform(y_tr)
                _ = y_scaler.transform(y_te)  # kept for symmetry

                base_pipe = make_pipeline_en(seed=seed + fold_id)

                best_loss_so_far = np.inf
                best_params_so_far = None
                trial_counter = count(0)

                # Local metrics container for THIS fold
                metrics_rows_local = []

                # Optional interim eval with current best hyperparams (no 1-SE)
                def _interim_eval(k_iter: int, best_plain: dict | None):
                    if not best_plain:
                        return
                    pipe_chk = make_pipeline_en(seed=seed + fold_id)
                    pipe_chk.set_params(**{f"model__{k}": v for k, v in best_plain.items()})
                    t_fitc = perf_counter()
                    pipe_chk.fit(X_tr, y_tr_z)
                    t_fitc = perf_counter() - t_fitc

                    yhat_tr = y_scaler.inverse_transform(pipe_chk.predict(X_tr))
                    yhat_te = y_scaler.inverse_transform(pipe_chk.predict(X_te))

                    r2_tr, rmse_tr, r2_te, rmse_te = [], [], [], []
                    for j in range(y_mat.shape[1]):
                        r2_tr.append(r2_score(y_tr[:, j], yhat_tr[:, j]))
                        rmse_tr.append(np.sqrt(mean_squared_error(y_tr[:, j], yhat_tr[:, j])))
                        r2_te.append(r2_score(y_te[:, j], yhat_te[:, j]))
                        rmse_te.append(np.sqrt(mean_squared_error(y_te[:, j], yhat_te[:, j])))
                    log(f"[PEEK @iter {k_iter}] Interim eval (no selection): "
                        f"TRAIN R2={np.mean(r2_tr):.4f}, RMSE={np.mean(rmse_tr):.4f} | "
                        f"TEST R2={np.mean(r2_te):.4f}, RMSE={np.mean(rmse_te):.4f} "
                        f"(fit {t_fitc:.2f}s)")

                # Objective: inner-CV mean MSE on z-scaled targets
                def obj(p_raw):
                    k = next(trial_counter) + 1
                    params = {
                        "alpha":    float(p_raw["alpha"]),
                        "l1_ratio": float(p_raw["l1_ratio"]),
                    }
                    base_pipe.set_params(
                        model__alpha=params["alpha"],
                        model__l1_ratio=params["l1_ratio"]
                    )
                    mses = -cross_val_score(
                        base_pipe, X_tr, y_tr_z,
                        cv=inner_cv, scoring="neg_mean_squared_error",
                        n_jobs=n_jobs_inner
                    )
                    loss = float(mses.mean())
                    loss_sem = float(mses.std(ddof=1) / np.sqrt(inner_cv.get_n_splits()))

                    nonlocal best_loss_so_far, best_params_so_far
                    if loss < best_loss_so_far:
                        best_loss_so_far = loss
                        best_params_so_far = params.copy()
                        log(f"Hyperopt iter {k}: **New best loss = {loss:.6f}**, params = {params}")

                    if (k % checkpoint_every) == 0:
                        log(f"Hyperopt iter {k}: current loss = {loss:.6f}, params = {params}")

                    if interim_eval_every and (k % interim_eval_every == 0):
                        _interim_eval(k, best_params_so_far)

                    return {"loss": loss, "loss_sem": loss_sem, **params, "status": STATUS_OK}

                # Run Hyperopt
                trials = Trials()
                fmin(
                    fn=obj,
                    space=hyperopt_space,
                    algo=tpe.suggest,
                    max_evals=int(hyperopt_evals),
                    trials=trials,
                    early_stop_fn=no_progress_loss(int(early_stop)),
                    rstate=np.random.default_rng(seed + fold_id),
                    show_progressbar=False
                )

                # Collect valid trials
                rows = []
                for tr in trials.trials:
                    res = tr.get("result", {})
                    if ("loss" in res) and np.isfinite(res["loss"]):
                        rows.append(res)
                res_df = pd.DataFrame(rows).dropna()
                assert len(res_df) > 0, f"Fold {fold_id}: no valid trials"

                # 1-SE (or min-loss) selection
                if use_one_se:
                    best_row = res_df.loc[res_df["loss"].idxmin()]
                    thr = float(best_row["loss"] + best_row["loss_sem"])
                    cands = res_df[res_df["loss"] <= thr].copy()
                    if prefer_stronger_in_1se:
                        # Prefer more shrinkage: alpha DESC, l1_ratio ASC
                        cands.sort_values(by=["alpha", "l1_ratio"], ascending=[False, True], inplace=True)
                    else:
                        cands.sort_values(by=["loss"], ascending=True, inplace=True)
                    chosen = cands.iloc[0]
                    log(f"1-SE chosen loss={chosen['loss']:.6f}, threshold={thr:.6f}")
                else:
                    chosen = res_df.loc[res_df["loss"].idxmin()]
                    log(f"Min loss chosen = {chosen['loss']:.6f}")

                best_params_plain = {
                    "alpha":    float(chosen["alpha"]),
                    "l1_ratio": float(chosen["l1_ratio"]),
                }
                best_params = {f"model__{k}": v for k, v in best_params_plain.items()}
                log("Parameters chosen for fold:\n" + json.dumps(to_json_compatible(best_params_plain), indent=2))

                # Final fit on TRAIN with chosen params
                t_fit = perf_counter()
                base_pipe.set_params(**best_params)
                base_pipe.fit(X_tr, y_tr_z)
                fit_time = perf_counter() - t_fit
                log(f"Fitted pipeline on TRAIN in {fit_time:.2f}s")

                # Predict (inverse-transform to original scale)
                yhat_tr = y_scaler.inverse_transform(base_pipe.predict(X_tr))
                yhat_te = y_scaler.inverse_transform(base_pipe.predict(X_te))

                # ---- Training performance (per target + macro) ----
                r2_tr_list, rmse_tr_list = [], []
                for j in range(y_mat.shape[1]):
                    r2_tr = r2_score(y_tr[:, j], yhat_tr[:, j])
                    rmse_tr = np.sqrt(mean_squared_error(y_tr[:, j], yhat_tr[:, j]))
                    r2_tr_list.append(r2_tr); rmse_tr_list.append(rmse_tr)
                    log(f"TRAIN target {j}: R2={r2_tr:.4f}, RMSE={rmse_tr:.4f}")
                log(f"TRAIN macro: R2={np.mean(r2_tr_list):.4f}, RMSE={np.mean(rmse_tr_list):.4f}")

                # Save TRAIN predictions
                df_tr = pd.DataFrame({'sample_id': X_df.index[tr_idx]})
                for j in range(y_mat.shape[1]):
                    df_tr[f'y_true_{j}'] = y_tr[:, j]
                    df_tr[f'y_pred_{j}'] = yhat_tr[:, j]
                    df_tr[f'resid_{j}']  = y_tr[:, j] - yhat_tr[:, j]
                tr_path = outdir / f"predictions_fold_{fold_id}_train.csv"
                df_tr.to_csv(tr_path, index=False)
                log(f"Saved train predictions to {tr_path}")

                # Save TEST predictions
                df_te = pd.DataFrame({'sample_id': X_df.index[te_idx]})
                for j in range(y_mat.shape[1]):
                    df_te[f'y_true_{j}'] = y_te[:, j]
                    df_te[f'y_pred_{j}'] = yhat_te[:, j]
                    df_te[f'resid_{j}']  = y_te[:, j] - yhat_te[:, j]
                te_path = outdir / f"predictions_fold_{fold_id}_test.csv"
                df_te.to_csv(te_path, index=False)
                log(f"Saved test predictions to {te_path}")

                # Per-target TRAIN/TEST metrics -> local container (this fold)
                for split, (yt, yp) in {"train": (y_tr, yhat_tr), "test": (y_te, yhat_te)}.items():
                    for j in range(y_mat.shape[1]):
                        r2  = r2_score(yt[:, j], yp[:, j])
                        mse = mean_squared_error(yt[:, j], yp[:, j])
                        rmse = float(np.sqrt(mse))
                        metrics_rows_local.append({
                            "fold": fold_id,
                            "split": split,
                            "target": f"{j}",
                            "R2": float(r2),
                            "MSE": float(mse),
                            "RMSE": rmse,
                            **best_params_plain
                        })
                        if split == "test":
                            log(f"TEST target {j}: R2={r2:.4f}, RMSE={rmse:.4f}")

                # Persist pipeline model
                model_path = outdir / f"pipeline_fold_{fold_id}.joblib"
                joblib.dump(base_pipe, model_path)
                log(f"Saved pipeline model to {model_path}")

                # Metadata per fold
                meta = {
                    "fold": fold_id,
                    "seed": seed + fold_id,
                    "best_params": best_params_plain,
                    "inner_best_loss": float(chosen["loss"]),
                    "inner_loss_sem": float(chosen.get("loss_sem", np.nan)),
                    "n_train": int(len(tr_idx)),
                    "n_test": int(len(te_idx)),
                    "interim_eval_every": int(interim_eval_every),
                    "checkpoint_every": int(checkpoint_every)
                }
                meta_clean = to_json_compatible(meta)
                meta_path = outdir / f"metadata_fold_{fold_id}.json"
                with open(meta_path, "w") as f:
                    json.dump(meta_clean, f, indent=2)
                log(f"Saved metadata to {meta_path}")

                # Save fold metrics to file (useful for debugging)
                metrics_df_fold = pd.DataFrame(metrics_rows_local)
                metrics_df_fold.to_csv(outdir / f"metrics_fold_{fold_id}.csv", index=False)

                log(f"END fold in {perf_counter() - t0:.2f}s")

                # Return both meta and fold metrics
                return {"meta": meta_clean, "metrics": metrics_df_fold}

    # Run folds in parallel (each fold logs to its own file)
    tasks = [(k+1, tr, te) for k, (tr, te) in enumerate(splits)]
    results = Parallel(n_jobs=n_jobs_outer)(
        delayed(_run_one_fold)(fold_id, tr_idx, te_idx)
        for fold_id, tr_idx, te_idx in tasks
    )

    # Assemble per-fold metadata
    all_meta = pd.DataFrame([r["meta"] for r in results])
    all_meta.to_csv(outdir / "all_metadata.csv", index=False)

    # Assemble metrics across folds
    metrics_df = pd.concat([r["metrics"] for r in results], ignore_index=True)
    metrics_df.to_csv(outdir / "metrics_all_folds.csv", index=False)

    # Summary (TEST only)
    test_df = metrics_df[metrics_df["split"] == "test"].copy()
    summary = (test_df.groupby("target", as_index=False)
               .agg(R2_mean=("R2", "mean"), R2_sd=("R2", "std"),
                    RMSE_mean=("RMSE", "mean"), RMSE_sd=("RMSE", "std")))
    summary.to_csv(outdir / "metrics_summary_test.csv", index=False)

    return all_meta, metrics_df, summary


In [10]:
OUT_EN_BASE = Path("../data/ml_output/mt_en_subset_panels_thr070/")
OUT_EN_BASE.mkdir(parents=True, exist_ok=True)

In [15]:
outer_cv = KFold(n_splits=10, shuffle=True, random_state=SEED)
inner_cv = KFold(n_splits=5,  shuffle=True, random_state=SEED)

In [12]:
HYPEROPT_SPACE_EN = {
    "alpha": hp.loguniform("alpha", np.log(3e-4), np.log(2.0)),  
    "l1_ratio": hp.uniform("l1_ratio", 0.3, 0.9), 
}

In [29]:
PANELS_EN = {
    "union_thr70":        X_union,
    "intersection_thr70": X_intersection,
    "xgb_only_thr70":     X_xgb_only,
    "en_only_thr70":      X_en_only,
}

summary_rows_en = []

for label, Xp in PANELS_EN.items():
    print(f"\n=== EN | Training panel: {label} ===")
    outdir_panel = OUT_EN_BASE / f"mt_en_{label}"
    outdir_panel.mkdir(parents=True, exist_ok=True)

    all_meta, metrics_df, summary_df = nested_cv_en_phase1(
        X_df=Xp,
        y_mat=y_mat,
        outer_cv=outer_cv,
        inner_cv=inner_cv,
        hyperopt_space=HYPEROPT_SPACE_EN,
        hyperopt_evals=100,
        early_stop=30,
        checkpoint_every=20,
        interim_eval_every=20,
        use_one_se=True,
        prefer_stronger_in_1se=True,
        n_jobs_inner=5, 
        n_jobs_outer=5,  
        outdir=outdir_panel,
        seed=SEED
    )

    s = summary_df.copy()
    s.insert(0, "panel", label)
    s["n_features"] = Xp.shape[1]
    summary_rows_en.append(s)

summary_all_en = pd.concat(summary_rows_en, ignore_index=True)
summary_all_en.to_csv(OUT_EN_BASE / "panels_performance_summary_en.csv", index=False)
print("\n[EN] Combined summary saved ->", OUT_EN_BASE / "panels_performance_summary_en.csv")
summary_all_en


=== EN | Training panel: union_thr70 ===

=== EN | Training panel: intersection_thr70 ===

=== EN | Training panel: xgb_only_thr70 ===

=== EN | Training panel: en_only_thr70 ===

[EN] Combined summary saved -> ..\data\ml_output\mt_en_subset_panels_thr070\panels_performance_summary_en.csv


,panel,target,R2_mean,R2_sd,RMSE_mean,RMSE_sd,n_features
0,union_thr70,0,0.858268,0.032730,0.030366,0.001970,436
1,union_thr70,1,0.864178,0.026898,1.491427,0.105916,436
2,intersection_thr70,0,0.746863,0.057073,0.040624,0.003370,72
3,intersection_thr70,1,0.671248,0.040812,2.332028,0.166354,72
4,xgb_only_thr70,0,0.838484,0.048202,0.032289,0.003701,114
5,xgb_only_thr70,1,0.844697,0.029902,1.595230,0.137920,114
6,en_only_thr70,0,0.809160,0.039153,0.035315,0.002490,250
7,en_only_thr70,1,0.822083,0.033910,1.708401,0.135440,250


In [13]:
PANELS_EN_NEW = {
    "xgb_all_thr70": X_xgb_all,
    "en_all_thr70":  X_en_all,
}

In [14]:
summary_rows_en_new = []

for label, Xp in PANELS_EN_NEW.items():
    print(f"\n=== EN | Training panel: {label} ===")
    outdir_panel = OUT_EN_BASE / f"mt_en_{label}"
    outdir_panel.mkdir(parents=True, exist_ok=True)

    all_meta, metrics_df, summary_df = nested_cv_en_phase1(
        X_df=Xp,
        y_mat=y_mat,
        outer_cv=outer_cv,
        inner_cv=inner_cv,
        hyperopt_space=HYPEROPT_SPACE_EN,
        hyperopt_evals=100,
        early_stop=30,
        checkpoint_every=20,
        interim_eval_every=20,
        use_one_se=True,
        prefer_stronger_in_1se=True,
        n_jobs_inner=5, 
        n_jobs_outer=5,  
        outdir=outdir_panel,
        seed=SEED
    )

    s = summary_df.copy()
    s.insert(0, "panel", label)
    s["n_features"] = Xp.shape[1]
    summary_rows_en_new.append(s)

summary_new_en = pd.concat(summary_rows_en_new, ignore_index=True)
summary_new_en.to_csv(OUT_EN_BASE / "panels_performance_summary_en_new.csv", index=False)
print("\n[EN] New panels summary saved ->", OUT_EN_BASE / "panels_performance_summary_en_new.csv")
summary_new_en


=== EN | Training panel: xgb_all_thr70 ===

=== EN | Training panel: en_all_thr70 ===

[EN] New panels summary saved -> ..\data\ml_output\mt_en_subset_panels_thr070\panels_performance_summary_en_new.csv


,panel,target,R2_mean,R2_sd,RMSE_mean,RMSE_sd,n_features
0,xgb_all_thr70,0,0.850667,0.039616,0.031087,0.002326,186
1,xgb_all_thr70,1,0.855748,0.029941,1.536159,0.132379,186
2,en_all_thr70,0,0.840899,0.034474,0.032224,0.002318,322
3,en_all_thr70,1,0.841052,0.036548,1.611862,0.172100,322


### XGBoost Multioutput

In [24]:
def make_pipeline_xgb(seed: int = 42, n_jobs_model: int = 1, use_multi_tree: bool = True) -> Pipeline:
    """
    Build a pipeline:
      - Standardize X (so Kernel SHAP / importances can use z-scored inputs if desired)
      - XGBRegressor with multi-output trees
    y-scaling is handled *outside* (per fold), exactly like EN, so we can invert predictions to the original scale.
    """
    params = dict(
        objective="reg:squarederror",
        tree_method="hist",
        n_estimators=600,
        learning_rate=0.05,
        max_depth=8,
        min_child_weight=4.0,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0,
        random_state=seed,
        n_jobs=n_jobs_model,
    )
    if use_multi_tree:
        params["multi_strategy"] = "multi_output_tree"

    return Pipeline(steps=[
        ("scale_x", StandardScaler(with_mean=True, with_std=True)),
        ("model", xgb.XGBRegressor(**params))
    ])

def nested_cv_xgb_phase1(
    X_df: pd.DataFrame,
    y_mat: np.ndarray,
    outer_cv: KFold,
    inner_cv: KFold,
    hyperopt_space: dict | None = None,
    hyperopt_evals: int = 100,
    early_stop: int = 30,
    checkpoint_every: int = 20,
    interim_eval_every: int = 20,
    use_one_se: bool = True,
    prefer_simpler_in_1se: bool = True,
    n_jobs_inner: int = 1,         
    n_jobs_outer: int = 1,  # folds in parallel
    outdir: Path | None = None,
    seed: int = 42,
    use_multi_tree: bool = True,
):
    """
    XGBoost nested CV with Hyperopt and EN-like IO/parallelization.
    - Per-fold logs: log_fold_{k}.txt
    - Inner-CV objective: mean MSE on *z-scaled* y
    - 1-SE selection (optional). If prefer_simpler_in_1se=True, tie-break towards simpler models.
    - Saves per fold:
        predictions (train/test), pipeline .joblib, metadata_fold_{k}.json, metrics_fold_{k}.csv
      and per panel:
        all_metadata.csv, metrics_all_folds.csv, metrics_summary_test.csv
    Returns: (all_meta_df, metrics_all_folds_df, summary_test_df)
    """
    assert outdir is not None, "Please provide an output directory."
    outdir.mkdir(parents=True, exist_ok=True)

    X = X_df.values.astype(np.float32)
    splits = list(outer_cv.split(X))

    # Default Hyperopt search space (robust ranges)
    if hyperopt_space is None:
        hyperopt_space = {
            "n_estimators":      hp.qloguniform("n_estimators", np.log(400), np.log(1200), 1),
            "learning_rate":     hp.loguniform("learning_rate", np.log(0.02), np.log(0.15)),
            "max_depth":         hp.choice("max_depth", [4, 6, 8, 10]),
            "min_child_weight":  hp.loguniform("min_child_weight", np.log(1.0), np.log(10.0)),
            "subsample":         hp.uniform("subsample", 0.6, 1.0),
            "colsample_bytree":  hp.uniform("colsample_bytree", 0.5, 1.0),
            "reg_lambda":        hp.loguniform("reg_lambda", np.log(1e-3), np.log(50.0)),
            "reg_alpha":         hp.loguniform("reg_alpha",  np.log(1e-5), np.log(5.0)),
            # optional:
            # "gamma":          hp.loguniform("gamma", np.log(1e-5), np.log(5.0)),
            # "colsample_bynode": hp.uniform("colsample_bynode", 0.5, 1.0),
        }

    def _simplicity_sort(df: pd.DataFrame) -> pd.DataFrame:
        """
        Prefer simpler models when several are within 1-SE:
        - shallower depth (ASC)
        - fewer trees (ASC)
        - larger min_child_weight (DESC)
        - larger reg_lambda / reg_alpha (DESC)
        - lower subsample / colsample_bytree (ASC)
        """
        keys = ["max_depth", "n_estimators", "min_child_weight", "reg_lambda", "reg_alpha", "subsample", "colsample_bytree"]
        for k in keys:
            if k not in df.columns:
                df[k] = np.nan
        # Sorting directions: depth↑bad, trees↑bad, min_child_weight↓bad, regs↓bad, subsample/colsample↑bad
        return df.sort_values(
            by=["max_depth", "n_estimators", "min_child_weight", "reg_lambda", "reg_alpha", "subsample", "colsample_bytree"],
            ascending=[True,       True,          False,             False,         False,        True,          True]
        )

    def _run_one_fold(fold_id, tr_idx, te_idx):
        with threadpool_limits(limits=1):
            with fold_logger(outdir, fold_id) as (log, log_path):
                t0 = perf_counter()
                log(f"START fold — train={len(tr_idx)}, test={len(te_idx)}, seed={seed + fold_id}, multi_tree={use_multi_tree}")

                X_tr, X_te = X[tr_idx], X[te_idx]
                y_tr, y_te = y_mat[tr_idx], y_mat[te_idx]

                # y-scaling outside pipeline to invert predictions to original scale
                y_scaler = StandardScaler()
                y_tr_z = y_scaler.fit_transform(y_tr)
                _ = y_scaler.transform(y_te)

                base_pipe = make_pipeline_xgb(seed=seed + fold_id, n_jobs_model=1, use_multi_tree=use_multi_tree)

                best_loss_so_far = np.inf
                best_params_so_far = None
                trial_counter = count(0)
                metrics_rows_local = []

                # Optional interim peek on TRAIN/TEST with current best params
                def _interim_eval(k_iter: int, best_plain: dict | None):
                    if not best_plain:
                        return
                    pipe_chk = make_pipeline_xgb(seed=seed + fold_id, n_jobs_model=1, use_multi_tree=use_multi_tree)
                    pipe_chk.set_params(**{f"model__{k}": v for k, v in best_plain.items()})
                    t_fitc = perf_counter()
                    pipe_chk.fit(X_tr, y_tr_z)
                    t_fitc = perf_counter() - t_fitc

                    yhat_tr = y_scaler.inverse_transform(pipe_chk.predict(X_tr))
                    yhat_te = y_scaler.inverse_transform(pipe_chk.predict(X_te))

                    r2_tr, rmse_tr, r2_te, rmse_te = [], [], [], []
                    for j in range(y_mat.shape[1]):
                        r2_tr.append(r2_score(y_tr[:, j], yhat_tr[:, j]))
                        rmse_tr.append(np.sqrt(mean_squared_error(y_tr[:, j], yhat_tr[:, j])))
                        r2_te.append(r2_score(y_te[:, j], yhat_te[:, j]))
                        rmse_te.append(np.sqrt(mean_squared_error(y_te[:, j], yhat_te[:, j])))
                    log(f"[PEEK @iter {k_iter}] Interim eval (no selection): "
                        f"TRAIN R2={np.mean(r2_tr):.4f}, RMSE={np.mean(rmse_tr):.4f} | "
                        f"TEST R2={np.mean(r2_te):.4f}, RMSE={np.mean(rmse_te):.4f} "
                        f"(fit {t_fitc:.2f}s)")

                # Objective for Hyperopt: inner-CV mean MSE on z-scaled targets
                def obj(p_raw):
                    k = next(trial_counter) + 1
                    p = {
                        "n_estimators":      int(round(float(p_raw["n_estimators"]))),
                        "learning_rate":     float(p_raw["learning_rate"]),
                        "max_depth":         int(p_raw["max_depth"]),
                        "min_child_weight":  float(p_raw["min_child_weight"]),
                        "subsample":         float(p_raw["subsample"]),
                        "colsample_bytree":  float(p_raw["colsample_bytree"]),
                        "reg_lambda":        float(p_raw["reg_lambda"]),
                        "reg_alpha":         float(p_raw["reg_alpha"]),
                    }
                    base_pipe.set_params(**{f"model__{k}": v for k, v in p.items()})

                    mses = -cross_val_score(
                        base_pipe, X_tr, y_tr_z,
                        cv=inner_cv, scoring="neg_mean_squared_error",
                        n_jobs=n_jobs_inner
                    )
                    loss = float(mses.mean())
                    loss_sem = float(mses.std(ddof=1) / np.sqrt(inner_cv.get_n_splits()))

                    nonlocal best_loss_so_far, best_params_so_far
                    if loss < best_loss_so_far:
                        best_loss_so_far = loss
                        best_params_so_far = p.copy()
                        log(f"Hyperopt iter {k}: **New best loss = {loss:.6f}**, params = {p}")

                    if (k % checkpoint_every) == 0:
                        log(f"Hyperopt iter {k}: current loss = {loss:.6f}, params = {p}")

                    if interim_eval_every and (k % interim_eval_every == 0):
                        _interim_eval(k, best_params_so_far)

                    return {"loss": loss, "loss_sem": loss_sem, **p, "status": STATUS_OK}

                # Hyperopt
                trials = Trials()
                fmin(
                    fn=obj,
                    space=hyperopt_space,
                    algo=tpe.suggest,
                    max_evals=int(hyperopt_evals),
                    trials=trials,
                    early_stop_fn=no_progress_loss(int(early_stop)),
                    rstate=np.random.default_rng(seed + fold_id),
                    show_progressbar=False
                )

                # Collect valid trials
                rows = []
                for tr in trials.trials:
                    res = tr.get("result", {})
                    if ("loss" in res) and np.isfinite(res["loss"]):
                        rows.append(res)
                res_df = pd.DataFrame(rows).dropna()
                assert len(res_df) > 0, f"Fold {fold_id}: no valid trials"

                # Selection: 1-SE or best loss
                if use_one_se:
                    best_row = res_df.loc[res_df["loss"].idxmin()]
                    thr = float(best_row["loss"] + best_row["loss_sem"])
                    cands = res_df[res_df["loss"] <= thr].copy()
                    if prefer_simpler_in_1se:
                        chosen = _simplicity_sort(cands).iloc[0]
                        log(f"1-SE chosen (simplest among near-best): loss={chosen['loss']:.6f}, thr={thr:.6f}")
                    else:
                        chosen = cands.sort_values("loss").iloc[0]
                        log(f"1-SE chosen (closest to best): loss={chosen['loss']:.6f}, thr={thr:.6f}")
                else:
                    chosen = res_df.loc[res_df["loss"].idxmin()]
                    log(f"Min loss chosen = {chosen['loss']:.6f}")

                best_params_plain = {
                    "n_estimators":     int(round(float(chosen["n_estimators"]))),
                    "learning_rate":    float(chosen["learning_rate"]),
                    "max_depth":        int(chosen["max_depth"]),
                    "min_child_weight": float(chosen["min_child_weight"]),
                    "subsample":        float(chosen["subsample"]),
                    "colsample_bytree": float(chosen["colsample_bytree"]),
                    "reg_lambda":       float(chosen["reg_lambda"]),
                    "reg_alpha":        float(chosen["reg_alpha"]),
                }
                best_params = {f"model__{k}": v for k, v in best_params_plain.items()}
                log("Parameters chosen for fold:\n" + json.dumps(to_json_compatible(best_params_plain), indent=2))

                # Final fit on TRAIN
                t_fit = perf_counter()
                base_pipe.set_params(**best_params)
                base_pipe.fit(X_tr, y_tr_z)
                fit_time = perf_counter() - t_fit
                log(f"Fitted pipeline on TRAIN in {fit_time:.2f}s")

                # Predict (invert to original scale)
                yhat_tr = y_scaler.inverse_transform(base_pipe.predict(X_tr))
                yhat_te = y_scaler.inverse_transform(base_pipe.predict(X_te))

                # ---- Training performance (per target + macro) ----
                r2_tr_list, rmse_tr_list = [], []
                for j in range(y_mat.shape[1]):
                    r2_tr = r2_score(y_tr[:, j], yhat_tr[:, j])
                    rmse_tr = np.sqrt(mean_squared_error(y_tr[:, j], yhat_tr[:, j]))
                    r2_tr_list.append(r2_tr); rmse_tr_list.append(rmse_tr)
                    log(f"TRAIN target {j}: R2={r2_tr:.4f}, RMSE={rmse_tr:.4f}")
                log(f"TRAIN macro: R2={np.mean(r2_tr_list):.4f}, RMSE={np.mean(rmse_tr_list):.4f}")

                # Save TRAIN predictions
                df_tr = pd.DataFrame({'sample_id': X_df.index[tr_idx]})
                for j in range(y_mat.shape[1]):
                    df_tr[f'y_true_{j}'] = y_tr[:, j]
                    df_tr[f'y_pred_{j}'] = yhat_tr[:, j]
                    df_tr[f'resid_{j}']  = y_tr[:, j] - yhat_tr[:, j]
                tr_path = outdir / f"predictions_fold_{fold_id}_train.csv"
                df_tr.to_csv(tr_path, index=False)
                log(f"Saved train predictions to {tr_path}")

                # Save TEST predictions
                df_te = pd.DataFrame({'sample_id': X_df.index[te_idx]})
                for j in range(y_mat.shape[1]):
                    df_te[f'y_true_{j}'] = y_te[:, j]
                    df_te[f'y_pred_{j}'] = yhat_te[:, j]
                    df_te[f'resid_{j}']  = y_te[:, j] - yhat_te[:, j]
                te_path = outdir / f"predictions_fold_{fold_id}_test.csv"
                df_te.to_csv(te_path, index=False)
                log(f"Saved test predictions to {te_path}")

                # Per-target TRAIN/TEST metrics -> local container
                for split, (yt, yp) in {"train": (y_tr, yhat_tr), "test": (y_te, yhat_te)}.items():
                    for j in range(y_mat.shape[1]):
                        r2  = r2_score(yt[:, j], yp[:, j])
                        mse = mean_squared_error(yt[:, j], yp[:, j])
                        rmse = float(np.sqrt(mse))
                        metrics_rows_local.append({
                            "fold": fold_id,
                            "split": split,
                            "target": f"{j}",
                            "R2": float(r2),
                            "MSE": float(mse),
                            "RMSE": rmse,
                            **best_params_plain
                        })
                        if split == "test":
                            log(f"TEST target {j}: R2={r2:.4f}, RMSE={rmse:.4f}")

                # Persist pipeline model
                model_path = outdir / f"pipeline_fold_{fold_id}.joblib"
                joblib.dump(base_pipe, model_path)
                log(f"Saved pipeline model to {model_path}")

                # Metadata per fold
                meta = {
                    "fold": fold_id,
                    "seed": seed + fold_id,
                    "multi_tree": use_multi_tree,
                    "best_params": best_params_plain,
                    "inner_best_loss": float(chosen["loss"]),
                    "inner_loss_sem": float(chosen.get("loss_sem", np.nan)),
                    "n_train": int(len(tr_idx)),
                    "n_test": int(len(te_idx)),
                    "interim_eval_every": int(interim_eval_every),
                    "checkpoint_every": int(checkpoint_every)
                }
                meta_clean = to_json_compatible(meta)
                meta_path = outdir / f"metadata_fold_{fold_id}.json"
                with open(meta_path, "w") as f:
                    json.dump(meta_clean, f, indent=2)
                log(f"Saved metadata to {meta_path}")

                # Save fold metrics to CSV
                metrics_df_fold = pd.DataFrame(metrics_rows_local)
                metrics_df_fold.to_csv(outdir / f"metrics_fold_{fold_id}.csv", index=False)

                log(f"END fold in {perf_counter() - t0:.2f}s")
                return {"meta": meta_clean, "metrics": metrics_df_fold}

    # Parallel over folds
    tasks = [(k+1, tr, te) for k, (tr, te) in enumerate(splits)]
    results = Parallel(n_jobs=n_jobs_outer)(
        delayed(_run_one_fold)(fold_id, tr_idx, te_idx)
        for fold_id, tr_idx, te_idx in tasks
    )

    # Aggregate outputs
    all_meta = pd.DataFrame([r["meta"] for r in results])
    all_meta.to_csv(outdir / "all_metadata.csv", index=False)

    metrics_df = pd.concat([r["metrics"] for r in results], ignore_index=True)
    metrics_df.to_csv(outdir / "metrics_all_folds.csv", index=False)

    test_df = metrics_df[metrics_df["split"] == "test"].copy()
    summary = (test_df.groupby("target", as_index=False)
               .agg(R2_mean=("R2", "mean"), R2_sd=("R2", "std"),
                    RMSE_mean=("RMSE", "mean"), RMSE_sd=("RMSE", "std")))
    summary.to_csv(outdir / "metrics_summary_test.csv", index=False)

    return all_meta, metrics_df, summary

In [25]:
# Base output dir per XGB
OUT_XGB_BASE = Path("../data/ml_output/xgb_multi_subset_panels_thr070/")
OUT_XGB_BASE.mkdir(parents=True, exist_ok=True)

In [26]:
# CV
outer_cv = KFold(n_splits=10, shuffle=True, random_state=SEED)
inner_cv = KFold(n_splits=5,  shuffle=True, random_state=SEED)

In [28]:
# Spazio di ricerca (identico/analogo a prima)
HYPEROPT_SPACE_XGB = {
    "n_estimators":      hp.qloguniform("n_estimators", np.log(400), np.log(1200), 1),
    "learning_rate":     hp.loguniform("learning_rate", np.log(0.02), np.log(0.15)),
    "max_depth":         hp.choice("max_depth", [2,3, 4]),
    "min_child_weight":  hp.loguniform("min_child_weight", np.log(1.0), np.log(50.0)),
    "subsample":         hp.uniform("subsample", 0.6, 1.0),
    "colsample_bytree":  hp.uniform("colsample_bytree", 0.5, 1.0),
    "reg_lambda":        hp.loguniform("reg_lambda", np.log(1e-1), np.log(50.0)),
    "reg_alpha":         hp.loguniform("reg_alpha",  np.log(1e-3), np.log(20.0)),
}

In [ ]:
# Pannelli (usa le stesse X già caricate)
PANELS_XGB = {
    "union_thr70":        X_union,
    "intersection_thr70": X_intersection,
    "xgb_only_thr70":     X_xgb_only,
    "en_only_thr70":      X_en_only,
}

In [64]:
summary_rows_xgb = []

for label, Xp in PANELS_XGB.items():
    print(f"\n=== XGB | Training panel: {label} ===")
    outdir_panel = OUT_XGB_BASE / f"xgb_multi_{label}"
    outdir_panel.mkdir(parents=True, exist_ok=True)

    all_meta, metrics_df, summary_df = nested_cv_xgb_phase1(
        X_df=Xp,
        y_mat=y_mat,
        outer_cv=outer_cv,
        inner_cv=inner_cv,
        hyperopt_space=HYPEROPT_SPACE_XGB,
        hyperopt_evals=50,
        early_stop=20,
        checkpoint_every=15,
        interim_eval_every=15,
        use_one_se=True,
        prefer_simpler_in_1se=True,
        n_jobs_inner=5,           
        n_jobs_outer=5,         
        outdir=outdir_panel,
        seed=SEED,
        use_multi_tree=True
    )

    s = summary_df.copy()
    s.insert(0, "panel", label)
    s["n_features"] = Xp.shape[1]
    summary_rows_xgb.append(s)

summary_all_xgb = pd.concat(summary_rows_xgb, ignore_index=True)
summary_all_xgb.to_csv(OUT_XGB_BASE / "panels_performance_summary_xgb.csv", index=False)
print("\n[XGB] Combined summary saved ->", OUT_XGB_BASE / "panels_performance_summary_xgb.csv")
summary_all_xgb


=== XGB | Training panel: union_thr70 ===

=== XGB | Training panel: intersection_thr70 ===

=== XGB | Training panel: xgb_only_thr70 ===

=== XGB | Training panel: en_only_thr70 ===

[XGB] Combined summary saved -> ..\data\ml_output\xgb_multi_subset_panels_thr070\panels_performance_summary_xgb.csv


,panel,target,R2_mean,R2_sd,RMSE_mean,RMSE_sd,n_features
0,union_thr70,0,0.862063,0.024874,0.030098,0.002449,436
1,union_thr70,1,0.867917,0.029575,1.471279,0.159231,436
2,intersection_thr70,0,0.799512,0.061954,0.035954,0.003770,72
3,intersection_thr70,1,0.769512,0.060313,1.929908,0.237247,72
4,xgb_only_thr70,0,0.859012,0.038276,0.030229,0.003391,114
5,xgb_only_thr70,1,0.871643,0.032429,1.445671,0.145250,114
6,en_only_thr70,0,0.822427,0.043217,0.034057,0.004114,250
7,en_only_thr70,1,0.827039,0.041335,1.680530,0.161274,250


In [29]:
XGB_NEW_PANELS = {
    "xgb_all_thr70": X_xgb_all,
    "en_all_thr70":  X_en_all,
}

In [30]:
summary_rows_xgb = []

for label, Xp in XGB_NEW_PANELS.items():
    print(f"\n=== XGB | Training panel: {label} ===")
    outdir_panel = OUT_XGB_BASE / f"xgb_multi_{label}"
    outdir_panel.mkdir(parents=True, exist_ok=True)

    all_meta, metrics_df, summary_df = nested_cv_xgb_phase1(
        X_df=Xp,
        y_mat=y_mat,
        outer_cv=outer_cv,
        inner_cv=inner_cv,
        hyperopt_space=HYPEROPT_SPACE_XGB,
        hyperopt_evals=50,
        early_stop=20,
        checkpoint_every=15,
        interim_eval_every=15,
        use_one_se=True,
        prefer_simpler_in_1se=True,
        n_jobs_inner=5,           
        n_jobs_outer=5,         
        outdir=outdir_panel,
        seed=SEED,
        use_multi_tree=True
    )

    s = summary_df.copy()
    s.insert(0, "panel", label)
    s["n_features"] = Xp.shape[1]
    summary_rows_xgb.append(s)

summary_all_xgb = pd.concat(summary_rows_xgb, ignore_index=True)
summary_all_xgb.to_csv(OUT_XGB_BASE / "panels_performance_summary_xgb.csv", index=False)
print("\n[XGB] Combined summary saved ->", OUT_XGB_BASE / "panels_performance_summary_xgb.csv")
summary_all_xgb


=== XGB | Training panel: xgb_all_thr70 ===

=== XGB | Training panel: en_all_thr70 ===

[XGB] Combined summary saved -> ..\data\ml_output\xgb_multi_subset_panels_thr070\panels_performance_summary_xgb.csv


,panel,target,R2_mean,R2_sd,RMSE_mean,RMSE_sd,n_features
0,xgb_all_thr70,0,0.867901,0.029455,0.029371,0.002797,186
1,xgb_all_thr70,1,0.868772,0.035824,1.460702,0.180533,186
2,en_all_thr70,0,0.835180,0.037436,0.032798,0.002640,322
3,en_all_thr70,1,0.835593,0.041599,1.636403,0.201246,322


### Performance check

In [19]:
import json, re, gc

# Ensure predictions exist 
def ensure_predictions_saved_en(
    X_df: pd.DataFrame,
    y_mat: np.ndarray,
    outer_cv,
    params_dir: Path,
    outdir: Path,
    seed: int = 42,
    overwrite: bool = False,
):
    """
    Recreate per-fold predictions if not present, using best params from metadata_fold_{k}.json.
    Assumes:
      - metadata_fold_{k}.json contains {"best_params": {"alpha": ..., "l1_ratio": ...}}
      - make_pipeline_en(seed=...) is available in scope
    """
    outdir.mkdir(parents=True, exist_ok=True)
    X = X_df.values.astype(np.float32)
    fold_iter = list(outer_cv.split(X))

    for fold_id, (tr_idx, te_idx) in enumerate(fold_iter, start=1):
        ftr = outdir / f"predictions_fold_{fold_id}_train.csv"
        fte = outdir / f"predictions_fold_{fold_id}_test.csv"
        if (ftr.exists() and fte.exists()) and not overwrite:
            continue

        meta_path = params_dir / f"metadata_fold_{fold_id}.json"
        if not meta_path.exists():
            raise FileNotFoundError(f"Missing metadata for fold {fold_id}: {meta_path}")
        with open(meta_path) as f:
            meta = json.load(f)
        bp = meta.get("best_params", {})
        if not {"alpha", "l1_ratio"}.issubset(bp):
            raise KeyError(f"metadata_fold_{fold_id}.json missing best_params alpha/l1_ratio")

        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y_mat[tr_idx], y_mat[te_idx]

        # y-scaling for inverse-transform
        y_scaler = StandardScaler(with_mean=True, with_std=True)
        y_tr_z = y_scaler.fit_transform(y_tr)

        pipe = make_pipeline_en(seed=seed + fold_id)
        pipe.set_params(model__alpha=float(bp["alpha"]), model__l1_ratio=float(bp["l1_ratio"]))
        pipe.fit(X_tr, y_tr_z)

        # inverse-transform to original scale
        yhat_tr = y_scaler.inverse_transform(pipe.predict(X_tr))
        yhat_te = y_scaler.inverse_transform(pipe.predict(X_te))

        pd.DataFrame({
            "sample_id": X_df.index[tr_idx],
            "y_true_0": y_tr[:, 0], "y_pred_0": yhat_tr[:, 0],
            "y_true_1": y_tr[:, 1], "y_pred_1": yhat_tr[:, 1],
        }).to_csv(ftr, index=False)

        pd.DataFrame({
            "sample_id": X_df.index[te_idx],
            "y_true_0": y_te[:, 0], "y_pred_0": yhat_te[:, 0],
            "y_true_1": y_te[:, 1], "y_pred_1": yhat_te[:, 1],
        }).to_csv(fte, index=False)

# Discover & load per-fold prediction files
def _discover_fold_pred_files(outdir: Path, which: str = 'test'):
    """
    Returns {fold_id: Path(...)} for predictions. Compatible with both flat files
    like 'predictions_fold_3_test.csv' and (if present) nested 'fold_3/predictions_test.csv'.
    """
    found = {}
    for fd in sorted(outdir.glob('fold_*')):
        if not fd.is_dir():
            continue
        kk = fd.name.split('_')[-1]
        cand1 = fd / f'predictions_{which}_fold_{kk}.csv'
        cand2 = fd / f'predictions_{which}.csv'
        if cand1.exists(): found[int(kk)] = cand1
        elif cand2.exists(): found[int(kk)] = cand2
    if not found:
        for fp in outdir.glob(f'predictions_fold_*_{which}.csv'):
            m = re.match(r'predictions_fold_(\d+)_', fp.name)
            if m: found[int(m.group(1))] = fp
    return dict(sorted(found.items()))

def load_predictions_by_fold(outdir: Path, which: str = 'test'):
    files = _discover_fold_pred_files(outdir, which=which)
    preds = {k: pd.read_csv(fp) for k, fp in files.items()}
    if not preds:
        raise FileNotFoundError(f'No {which} predictions found in {outdir}')
    return preds

# Plot: regplot True vs Pred per fold
def plot_reg_per_fold(preds_by_fold: dict,
                      target=0,
                      cols: int = 5,
                      figsize=(18, 7),
                      target_names: dict | None = None):
    """
    target: 0 or 1 (or '0'/'1'). Uses columns y_true_{t}, y_pred_{t}.
    """
    tkey = str(target)
    if target_names is None:
        target_names = {'0': 'Target 0', '1': 'Target 1'}
    disp = target_names.get(tkey, tkey)

    n_folds = len(preds_by_fold)
    rows = int(np.ceil(n_folds / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize, squeeze=False)
    axes = axes.ravel()

    y_true_all, y_pred_all = [], []
    for _, df in preds_by_fold.items():
        y_true_all.append(df[f'y_true_{tkey}'].values)
        y_pred_all.append(df[f'y_pred_{tkey}'].values)
    y_true_all = np.concatenate(y_true_all); y_pred_all = np.concatenate(y_pred_all)
    y_min = float(min(y_true_all.min(), y_pred_all.min()))
    y_max = float(max(y_true_all.max(), y_pred_all.max()))
    pad = 0.05 * (y_max - y_min) if y_max > y_min else 1.0
    lo, hi = y_min - pad, y_max + pad

    for ax, (k, df) in zip(axes, sorted(preds_by_fold.items())):
        yt = df[f'y_true_{tkey}'].values
        yp = df[f'y_pred_{tkey}'].values
        r2 = r2_score(yt, yp); mse = mean_squared_error(yt, yp)
        sns.regplot(x=yt, y=yp, ax=ax, scatter_kws=dict(s=15, alpha=0.7), line_kws=dict(lw=2))
        ax.plot([lo, hi], [lo, hi], ls='--', lw=1, color='black')
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
        ax.set_title(f'Fold {k} — R²={r2:.2f}, MSE={mse:.3f}', fontsize=10)
        ax.set_xlabel(f'{disp} (true)'); ax.set_ylabel(f'{disp} (pred)')

    for j in range(n_folds, rows*cols): axes[j].axis('off')
    fig.suptitle(f'Pred vs True per Fold — {disp}', y=1.02, fontsize=14)
    fig.tight_layout()
    return fig

# Plot: boxplot TRAIN vs TEST per target
def plot_box_train_test_metrics_per_target(outdir: Path,
                                           targets=(0, 1),
                                           target_names: dict | None = None,
                                           figsize=(12, 8)):
    """
    Reads metrics_all_folds.csv and plots boxplots for TRAIN vs TEST on R² and RMSE.
    Targets expected as '0'/'1' (we cast to string under the hood).
    """
    if target_names is None:
        target_names = {'0': 'Target 0', '1': 'Target 1'}

    m = pd.read_csv(outdir / 'metrics_all_folds.csv')
    # keep only TRAIN/TEST rows
    m = m[m['split'].isin(['train', 'test'])].copy()

    # normalize target to string
    m['target'] = m['target'].astype(str)
    keep_targets = [str(t) for t in targets]
    m = m[m['target'].isin(keep_targets)].copy()

    m['split'] = pd.Categorical(m['split'], categories=['train', 'test'], ordered=True)

    fig, axes = plt.subplots(nrows=len(keep_targets), ncols=2, figsize=figsize, squeeze=False)

    for i, t in enumerate(keep_targets):
        mt = m[m['target'] == t].copy()
        nice = target_names.get(t, t)
        ax_r2 = axes[i, 0]
        sns.boxplot(data=mt, x='split', y='R2', ax=ax_r2)
        sns.stripplot(data=mt, x='split', y='R2', ax=ax_r2, alpha=0.45, size=4, color='black')
        ax_r2.set_title(f"{nice} — R²"); ax_r2.set_xlabel("Split"); ax_r2.set_ylabel("R²")

        ax_rmse = axes[i, 1]
        sns.boxplot(data=mt, x='split', y='RMSE', ax=ax_rmse)
        sns.stripplot(data=mt, x='split', y='RMSE', ax=ax_rmse, alpha=0.45, size=4, color='black')
        ax_rmse.set_title(f"{nice} — RMSE"); ax_rmse.set_xlabel("Split"); ax_rmse.set_ylabel("RMSE")

        for ax in (ax_r2, ax_rmse):
            if ax.legend_: ax.legend_.remove()

    fig.tight_layout()
    return fig

def plot_compare_panels_test_metrics(
    panels_dirs: dict,
    targets=(0, 1),                   # puoi passare (0,1) o ("0","1")
    metrics=('R2', 'RMSE'),
    figsize=(14, 8),
    target_names=None,
    panel_labels=None,
    annotate_means: bool = False
):
    """
    Compare TEST performance across panels. Robust to missing files / type mismatches.
    """
    # Pretty names for targets
    if target_names is None:
        target_names = {'0': 'Propensity', '1': 'Flux Intensity'}

    # Pretty names for panels
    if panel_labels is None:
        panel_labels = {
            'union_thr70':        'Union',
            'intersection_thr70': 'Intersection',
            'xgb_only_thr70':     'XGB-only',
            'en_only_thr70':      'EN-only',
            "xgb_all_thr70":   "XGB-all",
            "en_all_thr70":    "EN-all",
        }

    # Normalize targets to strings
    targets_str = [str(t) for t in targets]

    rows, cols = len(metrics), len(targets_str)
    fig, axes = plt.subplots(rows, cols, figsize=figsize, squeeze=False)

    for j, t in enumerate(targets_str):
        dfs = []
        for label, pdir in panels_dirs.items():
            f = pdir / "metrics_all_folds.csv"
            if not f.exists():
                print(f"[WARN] Missing: {f}")
                continue
            df = pd.read_csv(f)
            # pick TEST rows; ignore 'scale' if not present
            if 'scale' in df.columns:
                df = df.query("split=='test' and scale=='original'").copy()
            else:
                df = df[df['split'] == 'test'].copy()

            # normalize target column to string
            df['target'] = df['target'].astype(str)
            df = df[df['target'] == t].copy()
            if df.empty:
                print(f"[WARN] No rows for target={t} in {f}")
                continue

            df["panel"] = panel_labels.get(label, label)
            dfs.append(df)

        if not dfs:
            # nothing to plot for this target; leave empty axes but label titles
            for i, met in enumerate(metrics):
                ax = axes[i, j]
                ax.set_title(f"{target_names.get(t, t)} — {met}")
                ax.set_xlabel("")
                ax.set_ylabel(met)
                ax.set_xlim(0, 1); ax.set_ylim(0, 1)  # so it doesn't autoscale weirdly
                ax.text(0.5, 0.5, "No data", ha='center', va='center', transform=ax.transAxes)
            continue

        data = pd.concat(dfs, ignore_index=True)

        for i, met in enumerate(metrics):
            ax = axes[i, j]
            sns.boxplot(data=data, x="panel", y=met, ax=ax, palette="Set2")
            sns.stripplot(data=data, x="panel", y=met, ax=ax, alpha=0.45, size=4, color='black')
            ax.set_title(f"{target_names.get(t, t)} — {met}", fontsize=16)
            ax.set_ylabel(met, fontsize=10)
            ax.set_xlabel("")  # keep tick labels, drop axis label
            ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right', fontsize=14)
            ax.grid(axis='y', linestyle='--', alpha=0.5)
            ax.tick_params(axis='y', labelsize=14)

            # Optional: annotate means above each box
            if annotate_means:
                means = data.groupby("panel")[met].mean()
                ymax = data[met].max()
                for xpos, pname in enumerate(means.index):
                    ax.text(xpos, ymax, f"{means[pname]:.3f}", ha='center', va='bottom', fontsize=9)

    fig.subplots_adjust(bottom=0.18, top=0.90, wspace=0.25, hspace=0.35)
    return fig

def savefig_and_close(fig, path, dpi=300):
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.close(fig)
    gc.collect()


#### Plots EN

In [13]:
OUT_EN_BASE = Path("../data/ml_output/mt_en_subset_panels_thr070/")
OUT_EN_BASE.mkdir(parents=True, exist_ok=True)

# Cartelle output per ogni pannello (coerenti con i risultati di training)
PANELS_EN = {
    "union_thr70":        OUT_EN_BASE / "mt_en_union_thr70",
    "intersection_thr70": OUT_EN_BASE / "mt_en_intersection_thr70",
    "xgb_only_thr70":     OUT_EN_BASE / "mt_en_xgb_only_thr70",
    "en_only_thr70":      OUT_EN_BASE / "mt_en_en_only_thr70",
    "xgb_all_thr70":      OUT_EN_BASE / "mt_en_xgb_all_thr70",
    "en_all_thr70":       OUT_EN_BASE / "mt_en_en_all_thr70",
}

# X matrices con LE STESSE CHIAVI di PANELS_EN
X_PANELS_EN = {
    "union_thr70":        X_union,
    "intersection_thr70": X_intersection,
    "xgb_only_thr70":     X_xgb_only,
    "en_only_thr70":      X_en_only,
    "xgb_all_thr70":      X_xgb_all,
    "en_all_thr70":       X_en_all,
}

TARGET_LABELS = {'0': 'Propensity', '1': 'Flux Intensity'}


In [18]:
# Recreate predictions if missing (reads best params from metadata_fold_k.json)
for label, outdir in PANELS_EN.items():
    print(f"[EN | {label}] ensuring predictions exist ...")
    ensure_predictions_saved_en(
        X_df=X_PANELS_EN[label],
        y_mat=y_mat,
        outer_cv=outer_cv,  
        params_dir=outdir,
        outdir=outdir,
        seed=SEED,
        overwrite=False
    )

for label, outdir in PANELS_EN.items():
    print(f"[EN | {label}] plotting per-fold regplots and train/test boxplots ...")
    plots_dir = outdir / "performance_plots"
    plots_dir.mkdir(parents=True, exist_ok=True)

    preds_test = load_predictions_by_fold(outdir, which='test')

    fig0 = plot_reg_per_fold(preds_by_fold=preds_test, target=0, cols=5, figsize=(16,6), target_names=TARGET_LABELS)
    savefig_and_close(fig0, plots_dir / 'regplot_per_fold_Propensity.png', dpi=300)

    fig1 = plot_reg_per_fold(preds_by_fold=preds_test, target=1, cols=5, figsize=(16,6), target_names=TARGET_LABELS)
    savefig_and_close(fig1, plots_dir / 'regplot_per_fold_FluxIntensity.png', dpi=300)

    fig_box = plot_box_train_test_metrics_per_target(outdir,
                                                     targets=(0,1),
                                                     target_names=TARGET_LABELS,
                                                     figsize=(11,7))
    savefig_and_close(fig_box, plots_dir / 'boxplots_by_target_r2_rmse.png', dpi=300)

fig_cmp = plot_compare_panels_test_metrics(
    panels_dirs=PANELS_EN,
    targets=(0, 1),
    metrics=('R2', 'RMSE'),
    figsize=(14, 8),
    target_names={'0': 'Propensity', '1': 'Flux Intensity'},  # se i CSV usano "0"/"1"
    annotate_means=False
)

CMP_DIR = OUT_EN_BASE / "panels_comparison_plots"
CMP_DIR.mkdir(parents=True, exist_ok=True)
savefig_and_close(fig_cmp, CMP_DIR / "compare_panels_test_R2_RMSE.png", dpi=300)

[EN | union_thr70] ensuring predictions exist ...
[EN | intersection_thr70] ensuring predictions exist ...
[EN | xgb_only_thr70] ensuring predictions exist ...
[EN | en_only_thr70] ensuring predictions exist ...
[EN | xgb_all_thr70] ensuring predictions exist ...
[EN | en_all_thr70] ensuring predictions exist ...
[EN | union_thr70] plotting per-fold regplots and train/test boxplots ...
[EN | intersection_thr70] plotting per-fold regplots and train/test boxplots ...
[EN | xgb_only_thr70] plotting per-fold regplots and train/test boxplots ...
[EN | en_only_thr70] plotting per-fold regplots and train/test boxplots ...
[EN | xgb_all_thr70] plotting per-fold regplots and train/test boxplots ...
[EN | en_all_thr70] plotting per-fold regplots and train/test boxplots ...


In [21]:
fig_cmp = plot_compare_panels_test_metrics(
    panels_dirs=PANELS_EN,
    targets=(0, 1),
    metrics=('R2', 'RMSE'),
    figsize=(14, 9),
    target_names={'0': 'Propensity', '1': 'Flux Intensity'},  # se i CSV usano "0"/"1"
    annotate_means=False
)

CMP_DIR = OUT_EN_BASE / "panels_comparison_plots"
CMP_DIR.mkdir(parents=True, exist_ok=True)
savefig_and_close(fig_cmp, CMP_DIR / "compare_panels_test_R2_RMSE.png", dpi=300)

#### Plots XGB

In [31]:
OUT_XGB_BASE = Path("../data/ml_output/xgb_multi_subset_panels_thr070/")
OUT_XGB_BASE.mkdir(parents=True, exist_ok=True)

# Cartelle output per ogni pannello (coerenti con i risultati di training)
PANELS_XGB_DIRS = {
    "union_thr70":        OUT_XGB_BASE / "xgb_multi_union_thr70",
    "intersection_thr70": OUT_XGB_BASE / "xgb_multi_intersection_thr70",
    "xgb_only_thr70":     OUT_XGB_BASE / "xgb_multi_xgb_only_thr70",
    "en_only_thr70":      OUT_XGB_BASE / "xgb_multi_en_only_thr70",
    "xgb_all_thr70":      OUT_XGB_BASE / "xgb_multi_xgb_all_thr70",
    "en_all_thr70":       OUT_XGB_BASE / "xgb_multi_en_all_thr70",
}

# X matrices con LE STESSE CHIAVI di PANELS_XGB
X_PANELS_XGB = {
    "union_thr70":        X_union,
    "intersection_thr70": X_intersection,
    "xgb_only_thr70":     X_xgb_only,
    "en_only_thr70":      X_en_only,
    "xgb_all_thr70":      X_xgb_all,
    "en_all_thr70":       X_en_all,
}

PANEL_LABELS = {
            'union_thr70':        'Union',
            'intersection_thr70': 'Intersection',
            'xgb_only_thr70':     'XGB-only',
            'en_only_thr70':      'EN-only',
            'xgb_all_thr70':      'XGB-all',
            'en_all_thr70':       'EN-all'
        }

TARGET_LABELS = {'0': 'Propensity', '1': 'Flux Intensity'}

In [32]:
for label, outdir in PANELS_XGB_DIRS.items():
    print(f"[XGB | {label}] plotting per-fold regplots and train/test boxplots ...")
    plots_dir = outdir / "performance_plots"
    plots_dir.mkdir(parents=True, exist_ok=True)

    preds_test = load_predictions_by_fold(outdir, which='test')

    fig0 = plot_reg_per_fold(preds_by_fold=preds_test, target=0, cols=5, figsize=(16,6), target_names=TARGET_LABELS)
    savefig_and_close(fig0, plots_dir / 'regplot_per_fold_Propensity.png', dpi=300)

    fig1 = plot_reg_per_fold(preds_by_fold=preds_test, target=1, cols=5, figsize=(16,6), target_names=TARGET_LABELS)
    savefig_and_close(fig1, plots_dir / 'regplot_per_fold_FluxIntensity.png', dpi=300)

    fig_box = plot_box_train_test_metrics_per_target(outdir,
                                                     targets=(0,1),
                                                     target_names=TARGET_LABELS,
                                                     figsize=(11,7))
    savefig_and_close(fig_box, plots_dir / 'boxplots_by_target_r2_rmse.png', dpi=300)

fig_cmp_xgb = plot_compare_panels_test_metrics(
    panels_dirs=PANELS_XGB_DIRS,
    targets=(0,1),
    metrics=('R2','RMSE'),
    figsize=(14,8),
    target_names=TARGET_LABELS,
    panel_labels=PANEL_LABELS
)
CMP_DIR_XGB = OUT_XGB_BASE / "panels_comparison"
CMP_DIR_XGB.mkdir(parents=True, exist_ok=True)
savefig_and_close(fig_cmp_xgb, CMP_DIR_XGB / "compare_panels_test_R2_RMSE_XGB_1.png", dpi=300)


[XGB | union_thr70] plotting per-fold regplots and train/test boxplots ...
[XGB | intersection_thr70] plotting per-fold regplots and train/test boxplots ...
[XGB | xgb_only_thr70] plotting per-fold regplots and train/test boxplots ...
[XGB | en_only_thr70] plotting per-fold regplots and train/test boxplots ...
[XGB | xgb_all_thr70] plotting per-fold regplots and train/test boxplots ...
[XGB | en_all_thr70] plotting per-fold regplots and train/test boxplots ...


#### Statistical Test

In [ ]:
from itertools import combinations
from scipy.stats import wilcoxon

def wilcoxon_pairwise(
    panels_dirs: dict[str, Path],
    metric: str = "R2",   
    target: int | str = 0,
    min_pairs: int = 5
) -> pd.DataFrame:
    """
    Runs paired Wilcoxon signed-rank tests for all pairs of panels on TEST split.
    Returns a tidy DataFrame with one row per comparison.

    Expected file per panel: <panel_dir>/metrics_all_folds.csv
    Columns used: ["fold","split","target", <metric>]
    """
    tgt = str(target)
    metric_lower = metric.lower()

    # Load per-panel series: index=fold, values=<metric>
    series_by_panel = {}
    for name, pdir in panels_dirs.items():
        f = Path(pdir) / "metrics_all_folds.csv"
        if not f.exists():
            print(f"[WARN] missing {f}, skipping panel '{name}'")
            continue
        m = pd.read_csv(f)
        if "fold" not in m.columns or metric not in m.columns:
            continue

        m = m[(m["split"] == "test")].copy()
        m["target"] = m["target"].astype(str)
        m = m[m["target"] == tgt]
        if m.empty:
            continue

        s = m.groupby("fold")[metric].mean().sort_index()
        series_by_panel[name] = s

    rows = []
    for a, b in combinations(series_by_panel.keys(), 2):
        sA, sB = series_by_panel[a], series_by_panel[b]
        common = sA.index.intersection(sB.index)
        if len(common) < min_pairs:
            rows.append({
                "panel_A": a, "panel_B": b,
                "n_pairs": int(len(common)),
                "wilcoxon_stat": np.nan, "p_value": np.nan,
                "median_diff": np.nan,
                "winner": None
            })
            continue

        x = sA.loc[common].values
        y = sB.loc[common].values
        d = x - y

        if np.allclose(d, 0.0):
            stat, p = 0.0, 1.0
        else:
            stat, p = wilcoxon(x, y, zero_method="wilcox", alternative="two-sided", method="auto")

        med_diff = float(np.median(d))

        if "rmse" in metric_lower or "mse" in metric_lower:
            # Lower is better
            if med_diff < 0:
                winner = a  # A has smaller RMSE → better
            elif med_diff > 0:
                winner = b  # B better
            else:
                winner = "tie"
        else:
            # Higher is better (R2 etc.)
            if med_diff > 0:
                winner = a
            elif med_diff < 0:
                winner = b
            else:
                winner = "tie"

        rows.append({
            "panel_A": a,
            "panel_B": b,
            "n_pairs": int(len(common)),
            "wilcoxon_stat": float(stat),
            "p_value": float(p),
            "median_diff": med_diff,
            "winner": winner
        })

    out = pd.DataFrame(rows)
    out = out.sort_values("p_value", na_position="last", ignore_index=True)
    return out
    return out


In [23]:
OUT_EN_BASE = Path("../data/ml_output/mt_en_subset_panels_thr070/")
PANELS_EN = {
    "union_thr70":        OUT_EN_BASE / "mt_en_union_thr70",
    "intersection_thr70": OUT_EN_BASE / "mt_en_intersection_thr70",
    "xgb_only_thr70":     OUT_EN_BASE / "mt_en_xgb_only_thr70",
    "en_only_thr70":      OUT_EN_BASE / "mt_en_en_only_thr70",
    "xgb_all_thr70":      OUT_EN_BASE / "mt_en_xgb_all_thr70",
    "en_all_thr70":       OUT_EN_BASE / "mt_en_en_all_thr70",
}

wilc_en_R2_tgt0 = wilcoxon_pairwise(PANELS_EN, metric="R2",  target=0)
wilc_en_R2_tgt1 = wilcoxon_pairwise(PANELS_EN, metric="R2",  target=1)
wilc_en_RMSE_0  = wilcoxon_pairwise(PANELS_EN, metric="RMSE", target=0)
wilc_en_RMSE_1  = wilcoxon_pairwise(PANELS_EN, metric="RMSE", target=1)

wilc_en_R2_tgt0.to_csv(OUT_EN_BASE / "panels_comparison/wilcoxon_pairwise_R2_target0.csv", index=False)
wilc_en_R2_tgt1.to_csv(OUT_EN_BASE / "panels_comparison/wilcoxon_pairwise_R2_target1.csv", index=False)
wilc_en_RMSE_0.to_csv(OUT_EN_BASE / "panels_comparison/wilcoxon_pairwise_RMSE_target0.csv", index=False)
wilc_en_RMSE_1.to_csv(OUT_EN_BASE / "panels_comparison/wilcoxon_pairwise_RMSE_target1.csv", index=False)

In [34]:
OUT_XGB_BASE = Path("../data/ml_output/xgb_multi_subset_panels_thr070/")
PANELS_XGB = {
    "union_thr70":        OUT_XGB_BASE / "xgb_multi_union_thr70",
    "intersection_thr70": OUT_XGB_BASE / "xgb_multi_intersection_thr70",
    "xgb_only_thr70":     OUT_XGB_BASE / "xgb_multi_xgb_only_thr70",
    "en_only_thr70":      OUT_XGB_BASE / "xgb_multi_en_only_thr70",
    "xgb_all_thr70":      OUT_XGB_BASE / "xgb_multi_xgb_all_thr70",
    "en_all_thr70":       OUT_XGB_BASE / "xgb_multi_en_all_thr70",
}

wilc_xgb_R2_tgt0 = wilcoxon_pairwise(PANELS_XGB, metric="R2",  target=0)
wilc_xgb_R2_tgt1 = wilcoxon_pairwise(PANELS_XGB, metric="R2",  target=1)
wilc_xgb_RMSE_0  = wilcoxon_pairwise(PANELS_XGB, metric="RMSE", target=0)
wilc_xgb_RMSE_1  = wilcoxon_pairwise(PANELS_XGB, metric="RMSE", target=1)

wilc_xgb_R2_tgt0.to_csv(OUT_XGB_BASE / "panels_comparison/wilcoxon_pairwise_R2_target0.csv", index=False)
wilc_xgb_R2_tgt1.to_csv(OUT_XGB_BASE / "panels_comparison/wilcoxon_pairwise_R2_target1.csv", index=False)
wilc_xgb_RMSE_0.to_csv(OUT_XGB_BASE / "panels_comparison/wilcoxon_pairwise_RMSE_target0.csv", index=False)
wilc_xgb_RMSE_1.to_csv(OUT_XGB_BASE / "panels_comparison/wilcoxon_pairwise_RMSE_target1.csv", index=False)


In [2]:
from itertools import combinations
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests


def wilcoxon_pairwise(
    panels_dirs: dict[str, Path],
    metric: str = "R2",
    target: int | str = 0,
    min_pairs: int = 5
) -> pd.DataFrame:
    """
    Run paired Wilcoxon signed-rank tests for all pairs of panel results (TEST split only).

    Each panel directory must contain a file: <panel_dir>/metrics_all_folds.csv
    with columns: ["fold", "split", "target", <metric>].

    The function compares the folds across panels for the given metric and target.
    It returns a tidy DataFrame with one row per comparison and includes:
      - p_value and Wilcoxon statistic
      - median difference
      - winner (higher = better for R2, lower = better for RMSE)
      - FDR-corrected q-value (Benjamini–Hochberg)
      - metric and target identifiers
    """
    tgt = str(target)
    metric_lower = metric.lower()

    # --- Load metrics from each panel directory
    series_by_panel = {}
    for name, pdir in panels_dirs.items():
        f = Path(pdir) / "metrics_all_folds.csv"
        if not f.exists():
            print(f"[WARN] missing {f}, skipping '{name}'")
            continue
        m = pd.read_csv(f)
        if "fold" not in m.columns or metric not in m.columns:
            continue

        # Keep only TEST split and the chosen target
        m = m.query("split == 'test'").copy()
        m["target"] = m["target"].astype(str)
        m = m[m["target"] == tgt]
        if m.empty:
            continue

        # Average metric per fold (just in case multiple rows per fold)
        s = m.groupby("fold")[metric].mean().sort_index()
        series_by_panel[name] = s

    # --- Pairwise Wilcoxon tests
    rows = []
    for a, b in combinations(series_by_panel.keys(), 2):
        sA, sB = series_by_panel[a], series_by_panel[b]
        common = sA.index.intersection(sB.index)

        if len(common) < min_pairs:
            # Not enough paired folds → skip
            rows.append({
                "panel_A": a, "panel_B": b, "n_pairs": int(len(common)),
                "wilcoxon_stat": np.nan, "p_value": np.nan,
                "median_diff": np.nan, "winner": None,
                "metric": metric, "target": tgt
            })
            continue

        x = sA.loc[common].values
        y = sB.loc[common].values
        d = x - y

        # If all differences are zero, p=1
        if np.allclose(d, 0.0):
            stat, p = 0.0, 1.0
        else:
            stat, p = wilcoxon(x, y, zero_method="wilcox", alternative="two-sided", method="auto")

        med_diff = float(np.median(d))

        # Winner logic: higher is better for R², lower for RMSE/MSE
        if "rmse" in metric_lower or "mse" in metric_lower:
            winner = a if med_diff < 0 else (b if med_diff > 0 else "tie")
        else:
            winner = a if med_diff > 0 else (b if med_diff < 0 else "tie")

        rows.append({
            "panel_A": a, "panel_B": b, "n_pairs": int(len(common)),
            "wilcoxon_stat": float(stat), "p_value": float(p),
            "median_diff": med_diff, "winner": winner,
            "metric": metric, "target": tgt
        })

    out = pd.DataFrame(rows)

    # --- FDR correction (Benjamini–Hochberg)
    if not out.empty and out["p_value"].notna().any():
        mask = out["p_value"].notna()
        _, qvals, _, _ = multipletests(out.loc[mask, "p_value"], method="fdr_bh")
        out.loc[mask, "fdr_bh"] = qvals
    else:
        out["fdr_bh"] = np.nan

    out = out.sort_values(["metric", "target", "p_value"], ignore_index=True)
    return out




In [3]:
# ElasticNet panels
OUT_EN_BASE = Path("../data/ml_output/mt_en_subset_panels_thr070/")
PANELS_EN = {
    "union_thr70":        OUT_EN_BASE / "mt_en_union_thr70",
    "intersection_thr70": OUT_EN_BASE / "mt_en_intersection_thr70",
    "xgb_only_thr70":     OUT_EN_BASE / "mt_en_xgb_only_thr70",
    "en_only_thr70":      OUT_EN_BASE / "mt_en_en_only_thr70",
    "xgb_all_thr70":      OUT_EN_BASE / "mt_en_xgb_all_thr70",
    "en_all_thr70":       OUT_EN_BASE / "mt_en_en_all_thr70",
}

EN_COMP_DIR = OUT_EN_BASE / "panels_comparison"
EN_COMP_DIR.mkdir(parents=True, exist_ok=True)

dfs_en = []
for metric in ["R2", "RMSE"]:
    for tgt in [0, 1]:
        dfs_en.append(wilcoxon_pairwise(PANELS_EN, metric=metric, target=tgt))
en_all = pd.concat(dfs_en, ignore_index=True)
en_all.to_csv(EN_COMP_DIR / "wilcoxon_pairwise_ALL.csv", index=False)


# XGBoost panels
OUT_XGB_BASE = Path("../data/ml_output/xgb_multi_subset_panels_thr070/")
PANELS_XGB = {
    "union_thr70":        OUT_XGB_BASE / "xgb_multi_union_thr70",
    "intersection_thr70": OUT_XGB_BASE / "xgb_multi_intersection_thr70",
    "xgb_only_thr70":     OUT_XGB_BASE / "xgb_multi_xgb_only_thr70",
    "en_only_thr70":      OUT_XGB_BASE / "xgb_multi_en_only_thr70",
    "xgb_all_thr70":      OUT_XGB_BASE / "xgb_multi_xgb_all_thr70",
    "en_all_thr70":       OUT_XGB_BASE / "xgb_multi_en_all_thr70",
}

XGB_COMP_DIR = OUT_XGB_BASE / "panels_comparison"
XGB_COMP_DIR.mkdir(parents=True, exist_ok=True)

dfs_xgb = []
for metric in ["R2", "RMSE"]:
    for tgt in [0, 1]:
        dfs_xgb.append(wilcoxon_pairwise(PANELS_XGB, metric=metric, target=tgt))
xgb_all = pd.concat(dfs_xgb, ignore_index=True)
xgb_all.to_csv(XGB_COMP_DIR / "wilcoxon_pairwise_ALL.csv", index=False)


### SHAP

In [ ]:
import json, gc, re
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits


def get_fi_dirs(panel_dir: Path) -> dict:
    """
    Create/return standard subfolders:
      <panel_dir>/feature_importance/{shap,plots}
    """
    fi_base   = panel_dir / "feature_importance"
    shap_dir  = fi_base / "shap"
    plots_dir = fi_base / "plots"
    for d in (shap_dir, plots_dir):
        d.mkdir(parents=True, exist_ok=True)
    return {"base": fi_base, "shap": shap_dir, "plots": plots_dir}

def savefig_and_close(fig, path: Path, dpi=150):
    """Save a figure and free memory."""
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig); gc.collect()

def _slugify(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", s.strip())

def load_gene_symbol_map(csv_path: str | Path) -> dict:
    """Load ENSG→symbol mapping from CSV (expected columns: gene_id, gene_name)."""
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=["gene_id"]).drop_duplicates(subset=["gene_id"], keep="first")
    return dict(zip(df["gene_id"].astype(str), df["gene_name"].astype(str)))

def build_feature_name_list(ensg_cols, mapping: dict, disambiguate: bool = True) -> list[str]:
    """
    Build display feature names (symbol if available, else ENSG).
    If disambiguate=True, append (ENSG) when a symbol occurs multiple times.
    """
    prelim = [mapping.get(g, g) for g in ensg_cols]
    if not disambiguate:
        return prelim
    counts = {}
    for name in prelim:
        counts[name] = counts.get(name, 0) + 1
    out = []
    for name, g in zip(prelim, ensg_cols):
        if counts.get(name, 0) > 1 and name != g:
            out.append(f"{name} ({g})")
        else:
            out.append(name)
    return out

def _normalize_multitask_shap(shap_vals, n_targets: int = 2):
    """
    Normalize SHAP output for multi-output models into a list [sv0, sv1],
    each with shape (n_samples, n_features).
    """
    if isinstance(shap_vals, (list, tuple)):
        if len(shap_vals) != n_targets:
            raise ValueError(f"Expected {n_targets} targets, got {len(shap_vals)}.")
        sv = [np.asarray(v) for v in shap_vals]
        if any(v.ndim != 2 for v in sv):
            raise ValueError("Each target should be 2D (n_samples, n_features).")
        return sv
    arr = np.asarray(shap_vals)
    if arr.ndim == 3 and arr.shape[-1] == n_targets:
        return [arr[..., i] for i in range(n_targets)]
    raise ValueError(f"Unsupported SHAP shape {arr.shape}.")

# ---------- SHAP per-fold for XGB: load saved pipelines ----------

def _shap_one_fold_xgb(
    fold_id: int,
    X: np.ndarray,
    X_index,
    tr_idx: np.ndarray,
    te_idx: np.ndarray,
    panel_dir: Path,
    shap_dir: Path,
    which: str,
    background_subsample: int,
    nsamples_kernel: int,
    seed: int,
    gene_names: list[str]
):
    """
    Compute SHAP for a single fold and ALWAYS save CSV with header = gene_names.

    Artifacts written:
      - shap_fold{fold_id}_A.csv / shap_fold{fold_id}_B.csv
        (index = sample_id, columns = gene_names)
      - shap_meanabs_fold{fold_id}_{A|B}.csv (columns: fold, gene, target, mean_abs_shap)
      - shap_expected_value_fold{fold_id}.json

    Returns a long DataFrame with per-fold mean|SHAP|.
    """
    with threadpool_limits(limits=1):  # avoid nested BLAS oversubscription
        pipe_path = panel_dir / f"pipeline_fold_{fold_id}.joblib"
        if not pipe_path.exists():
            raise FileNotFoundError(f"Missing model for fold {fold_id}: {pipe_path}")
        pipe = joblib.load(pipe_path)

        # Choose eval split
        X_tr, X_te = X[tr_idx], X[te_idx]
        if which == "train":
            X_eval = X_tr; idx_eval = X_index[tr_idx]
        else:
            X_eval = X_te; idx_eval = X_index[te_idx]

        # Background on TRAIN (optional subsample)
        X_bg = X_tr
        if background_subsample and len(X_bg) > background_subsample:
            rng = np.random.default_rng(seed + fold_id)
            sel = rng.choice(len(X_bg), size=background_subsample, replace=False)
            X_bg = X_bg[sel]

        # KernelExplainer on pipeline's predict (includes any scaler inside the pipeline)
        f = pipe.predict  # returns shape (n_samples, 2)
        expl = shap.KernelExplainer(model=f, data=X_bg, link="identity")

        # SHAP values
        sv_raw = expl.shap_values(X_eval, nsamples=nsamples_kernel)
        sv_list = _normalize_multitask_shap(sv_raw, n_targets=2)  # [sv_A, sv_B]

        # Expected values
        ev = expl.expected_value
        if isinstance(ev, (list, tuple)):
            evA, evB = float(ev[0]), float(ev[1])
        else:
            ev = np.asarray(ev).reshape(-1)
            evA, evB = float(ev[0]), float(ev[1])
        with open(shap_dir / f"shap_expected_value_fold{fold_id}.json", "w") as f_out:
            json.dump({"fold": fold_id, "expected_value_A": evA, "expected_value_B": evB}, f_out, indent=2)

        # Save SHAP matrices with proper headers + per-fold mean|SHAP|
        all_rows = []
        for (sv, tgt_name) in zip(sv_list, ["A", "B"]):
            df_sv = pd.DataFrame(sv.astype(np.float32), index=idx_eval, columns=gene_names)
            df_sv.to_csv(shap_dir / f"shap_fold{fold_id}_{tgt_name}.csv", index=True)

            meanabs = np.mean(np.abs(sv), axis=0).astype(np.float32)
            fold_imp = pd.DataFrame({
                "fold": fold_id,
                "gene": gene_names,
                "target": tgt_name,
                "mean_abs_shap": meanabs
            })
            fold_imp.to_csv(shap_dir / f"shap_meanabs_fold{fold_id}_{tgt_name}.csv", index=False)
            all_rows.append(fold_imp)

        return pd.concat(all_rows, ignore_index=True)

def shap_per_fold_xgb_kernel(
    X_df: pd.DataFrame,
    y_mat: np.ndarray,       # included for symmetry/checks
    outer_cv,
    panel_dir: Path,
    shap_outdir: Path,
    which: str = "test",
    background_subsample: int = 400,
    nsamples_kernel: int = 1024,
    seed: int = 42,
    n_jobs_outer: int = 6
) -> pd.DataFrame:
    """
    Compute SHAP for each fold using saved pipelines and write:
      - shap_fold{k}_{A|B}.csv (header = gene_names)
      - shap_meanabs_fold{k}_{A|B}.csv
      - shap_expected_value_fold{k}.json
      - shap_meanabs_crossfold.csv

    Returns cross-fold summary with columns:
      gene, target, mean_abs_shap_mean, mean_abs_shap_sd, n_folds.
    """
    shap_outdir.mkdir(parents=True, exist_ok=True)

    X = X_df.values.astype(np.float32)
    X_index = X_df.index.to_numpy()
    gene_names = list(X_df.columns)

    folds = list(outer_cv.split(X))
    results = Parallel(n_jobs=n_jobs_outer)(
        delayed(_shap_one_fold_xgb)(
            fold_id=k+1,
            X=X,
            X_index=X_index,
            tr_idx=tr,
            te_idx=te,
            panel_dir=panel_dir,
            shap_dir=shap_outdir,
            which=which,
            background_subsample=background_subsample,
            nsamples_kernel=nsamples_kernel,
            seed=seed,
            gene_names=gene_names
        )
        for k, (tr, te) in enumerate(folds)
    )

    # Cross-fold summary
    all_meanabs = pd.concat(results, ignore_index=True)
    cross = (all_meanabs
             .groupby(["gene", "target"], as_index=False)
             .agg(mean_abs_shap_mean=("mean_abs_shap", "mean"),
                  mean_abs_shap_sd=("mean_abs_shap", "std"),
                  n_folds=("mean_abs_shap", "count"))
             .sort_values(["target", "mean_abs_shap_mean"], ascending=[True, False]))
    cross.to_csv(shap_outdir / "shap_meanabs_crossfold.csv", index=False)
    return cross

# ---------- helpers for beeswarm coloring (z-scores) ----------

def _gather_eval_z_color_xgb(X_df, outer_cv, panel_dir: Path, which="test"):
    """
    Reload the pipeline for each fold and return {fold_id: X_eval_z} where X_eval_z
    are standardized features (z-scores). If the pipeline has 'scale_x', use it; 
    otherwise fit StandardScaler on TRAIN of that fold.
    """
    X = X_df.values.astype(np.float32)
    out = {}
    for k, (tr_idx, te_idx) in enumerate(outer_cv.split(X), start=1):
        pipe = joblib.load(panel_dir / f"pipeline_fold_{k}.joblib")
        X_tr, X_te = X[tr_idx], X[te_idx]
        X_eval = X_tr if which == "train" else X_te

        if hasattr(pipe, "named_steps") and "scale_x" in pipe.named_steps:
            z = pipe.named_steps["scale_x"].transform(X_eval)
        else:
            sc = StandardScaler(with_mean=True, with_std=True).fit(X_tr)
            z = sc.transform(X_eval)
        out[k] = z
    return out

def plot_shap_beeswarm_all_folds(
    X_df, outer_cv, panel_dir: Path, shap_dir: Path,
    target="A", feature_names=None,
    top_k=20, max_points=5000, seed=42, outpath: Path=None
):
    """
    Global beeswarm stacking across folds. Colors use z-scored features.
    """
    eval_z = _gather_eval_z_color_xgb(X_df, outer_cv, panel_dir, which="test")
    mats, feats = [], []
    for fold_id in sorted(eval_z.keys()):
        sv_path = shap_dir / f"shap_fold{fold_id}_{target}.csv"
        sv = pd.read_csv(sv_path, index_col=0).values.astype(np.float32)
        mats.append(sv); feats.append(eval_z[fold_id])

    SHAP_all = np.vstack(mats)
    FEAT_all = np.vstack(feats)

    if max_points and SHAP_all.shape[0] > max_points:
        rng = np.random.default_rng(seed)
        sel = rng.choice(SHAP_all.shape[0], size=max_points, replace=False)
        SHAP_all = SHAP_all[sel]; FEAT_all = FEAT_all[sel]

    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        SHAP_all, FEAT_all,
        feature_names=(feature_names if feature_names is not None else list(X_df.columns)),
        plot_type="dot", max_display=top_k, show=False
    )
    plt.yticks(fontsize=22)
    plt.xticks(fontsize=18)
    plt.xlabel(None)
    cbar = plt.gcf().axes[-1]
    cbar.tick_params(labelsize=16)
    cbar.set_ylabel("Expression (z-score)", fontsize=16)
    plt.tight_layout()
    if outpath is not None:
        savefig_and_close(plt.gcf(), outpath, dpi=150)
    return plt.gcf()

def plot_shap_bar_crossfold(
    shap_cross_csv: Path,
    target: str = "A",
    top_k: int = 20,
    gene_map_csv: str | Path | None = None,
    disambiguate_symbols: bool = True,
    outpath: Path | None = None
):
    """
    Bar plot of top-k genes by mean|SHAP| with ±SD across folds.
    X tick labels use SYMBOLS (or ENSG if missing).
    """
    cf = pd.read_csv(shap_cross_csv)
    cf_t = (cf[cf["target"] == target]
            .sort_values("mean_abs_shap_mean", ascending=False)
            .head(top_k).copy())

    mapping = load_gene_symbol_map(gene_map_csv) if gene_map_csv else {}
    genes_en = cf_t["gene"].astype(str).tolist()
    names = build_feature_name_list(genes_en, mapping, disambiguate=disambiguate_symbols)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(np.arange(len(cf_t)), cf_t["mean_abs_shap_mean"],
           yerr=cf_t["mean_abs_shap_sd"], capsize=3)
    ax.set_xlabel(None)
    ax.set_ylabel("mean |SHAP|", fontsize=16)
    ax.set_xticks(np.arange(len(cf_t)))
    ax.set_xticklabels(names, rotation=75, ha="right", fontsize=14)
    ax.tick_params(axis='y', labelsize=18)
    plt.tight_layout()
    if outpath is not None:
        savefig_and_close(fig, outpath, dpi=150)
    return fig

def plot_shap_stability_dot(
    shap_cross_csv: Path,
    target: str = "A",
    top_k: int = 20,
    gene_map_csv: str | Path | None = None,
    disambiguate_symbols: bool = True,
    outpath: Path | None = None
):
    """
    Dot + whisker (mean±SD) for the top-k genes to show cross-fold stability.
    X tick labels use SYMBOLS (or ENSG if missing).
    """
    cf = pd.read_csv(shap_cross_csv)
    cf_t = (cf[cf["target"] == target]
            .sort_values("mean_abs_shap_mean", ascending=True)
            .tail(top_k).copy())

    mapping = load_gene_symbol_map(gene_map_csv) if gene_map_csv else {}
    genes_en = cf_t["gene"].astype(str).tolist()
    names = build_feature_name_list(genes_en, mapping, disambiguate=disambiguate_symbols)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.errorbar(x=np.arange(len(cf_t)),
                y=cf_t["mean_abs_shap_mean"],
                yerr=cf_t["mean_abs_shap_sd"],
                fmt="o", capsize=3)
    ax.set_xticks(np.arange(len(cf_t)))
    ax.set_xticklabels(names, rotation=75, ha="right", fontsize=14)
    ax.set_xlabel(None)
    ax.set_ylabel("mean |SHAP|", fontsize=16)
    ax.tick_params(axis='y', labelsize=18)
    plt.tight_layout()
    if outpath is not None:
        savefig_and_close(fig, outpath, dpi=150)
    return fig

def plot_shap_biplot_AB(
    shap_cross_csv: Path,
    target_labels: dict,
    gene_map_csv: str | Path | None = None,
    log_scale: bool = False,
    min_value: float = 1e-6,
    pad_frac: float = 0.05,
    cap_by_percentile: float | None = None,
    annotate_top_k: int = 0,
    outpath: Path | None = None
):
    """
    Biplot mean|SHAP| (A vs B) with axes taken from actual min/max plus padding.
    - If log_scale=True, axes are log-log and values are clipped to min_value.
    - cap_by_percentile (e.g. 99.5) limits outlier influence on axis range.
    - annotate_top_k optionally labels top genes (by geometric mean) with SYMBOLS.
    """
    cf = pd.read_csv(shap_cross_csv)
    A = cf[cf["target"]=="A"][["gene","mean_abs_shap_mean"]].rename(columns={"mean_abs_shap_mean":"A_mean"})
    B = cf[cf["target"]=="B"][["gene","mean_abs_shap_mean"]].rename(columns={"mean_abs_shap_mean":"B_mean"})
    M = A.merge(B, on="gene", how="outer").fillna(0.0)

    genes_en = M["gene"].astype(str).tolist()
    mapping = load_gene_symbol_map(gene_map_csv) if gene_map_csv else {}
    symbols = build_feature_name_list(genes_en, mapping, disambiguate=True)

    x = M["A_mean"].to_numpy(dtype=float)
    y = M["B_mean"].to_numpy(dtype=float)

    # Optional percentile capping
    if cap_by_percentile is not None:
        p = float(cap_by_percentile)
        lo_p, hi_p = 100 - p, p
        x = np.clip(x, np.percentile(x, lo_p), np.percentile(x, hi_p))
        y = np.clip(y, np.percentile(y, lo_p), np.percentile(y, hi_p))

    if log_scale:
        x = np.clip(x, min_value, None)
        y = np.clip(y, min_value, None)

    def _limits(arr, log=False):
        lo, hi = float(arr.min()), float(arr.max())
        if log:
            return lo / (1 + pad_frac), hi * (1 + pad_frac)
        rng = max(hi - lo, 1e-12)
        pad = pad_frac * rng
        return lo - pad, hi + pad

    x_lo, x_hi = _limits(x, log=log_scale)
    y_lo, y_hi = _limits(y, log=log_scale)

    fig, ax = plt.subplots(figsize=(7,7))
    ax.scatter(x, y, s=25, alpha=0.7)

    if log_scale:
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlim(max(min_value, x_lo), x_hi)
        ax.set_ylim(max(min_value, y_lo), y_hi)
        lo_diag = max(min_value, min(ax.get_xlim()[0], ax.get_ylim()[0]))
        hi_diag = max(ax.get_xlim()[1], ax.get_ylim()[1])
        ax.plot([lo_diag, hi_diag], [lo_diag, hi_diag], ls="--", lw=1.2, c="k", alpha=0.6)
    else:
        ax.set_xlim(x_lo, x_hi); ax.set_ylim(y_lo, y_hi)
        lo_diag = min(ax.get_xlim()[0], ax.get_ylim()[0])
        hi_diag = max(ax.get_xlim()[1], ax.get_ylim()[1])
        ax.plot([lo_diag, hi_diag], [lo_diag, hi_diag], ls="--", lw=1.2, c="k", alpha=0.6)

    ax.set_xlabel(f"mean |SHAP| ({target_labels.get('A','A')})", fontsize=16)
    ax.set_ylabel(f"mean |SHAP| ({target_labels.get('B','B')})", fontsize=16)
    ax.tick_params(axis='both', labelsize=14)
    ax.grid(True, which="both", linestyle="--", alpha=0.3)
    plt.tight_layout()

    # Optional annotations
    if annotate_top_k and annotate_top_k > 0:
        gm = np.sqrt(x * y)
        idx = np.argsort(gm)[::-1][:int(annotate_top_k)]
        for i in idx:
            ax.annotate(symbols[i], (x[i], y[i]),
                        xytext=(6,3), textcoords="offset points",
                        fontsize=9, color="black")

    if outpath is not None:
        savefig_and_close(fig, outpath, dpi=180)
    return fig

# ---------- end-to-end runner (updated calls) ----------

def run_shap_for_panel(
    X_df: pd.DataFrame,
    y_mat: np.ndarray,
    outer_cv,
    panel_dir: Path,                 # must contain pipeline_fold_{k}.joblib
    target_labels: dict | None = None,   # {'A':'Propensity','B':'Flux Intensity'}
    gene_map_csv: str | Path | None = None,
    top_k: int = 20,
    seed: int = 42,
    background_subsample: int = 400,
    nsamples_kernel: int = 1024,
    n_jobs_outer: int = 6,
    do_biplot_log: bool = True,
    biplot_min_value: float = 1e-6,
    biplot_pad_frac: float = 0.05,          # NEW
    biplot_cap_percentile: float | None = None,  # NEW (e.g. 99.5)
    biplot_annotate_top_k: int = 0          # NEW
):
    """
    Compute SHAP via KernelExplainer from saved XGB pipelines and produce:
      - beeswarm per target (stacked across folds)
      - bar ±SD (top-k) per target [SYMBOL labels]
      - stability (dot+whisker) per target [SYMBOL labels]
      - biplot A vs B (linear/log, min/max padded, optional percentile cap)
    """
    def _ts(): return datetime.now().strftime("%H:%M:%S")
    def _log(msg): print(f"[{_ts()}] {msg}")

    if target_labels is None:
        target_labels = {'A': 'A', 'B': 'B'}
    tslugA = _slugify(target_labels['A'])
    tslugB = _slugify(target_labels['B'])

    D = get_fi_dirs(panel_dir)
    shap_dir, plots_dir = D["shap"], D["plots"]

    # mapping for beeswarm feature_names
    mapping = {}
    if gene_map_csv is not None and Path(gene_map_csv).exists():
        _log(f"Loading gene map: {gene_map_csv}")
        mapping = load_gene_symbol_map(gene_map_csv)
    feature_names = build_feature_name_list(X_df.columns.tolist(), mapping, disambiguate=True)

    _log(f"SHAP (XGB/Kernel) — bg={background_subsample}, nsamples={nsamples_kernel}, folds={outer_cv.get_n_splits()}")
    cross = shap_per_fold_xgb_kernel(
        X_df=X_df, y_mat=y_mat, outer_cv=outer_cv,
        panel_dir=panel_dir, shap_outdir=shap_dir,
        which="test", background_subsample=background_subsample,
        nsamples_kernel=nsamples_kernel, seed=seed, n_jobs_outer=n_jobs_outer
    )
    _log(f"Saved cross-fold summary -> {shap_dir / 'shap_meanabs_crossfold.csv'} (rows={len(cross)})")

    # Plots
    _log(f"Plotting beeswarm ({target_labels['A']}) ...")
    plot_shap_beeswarm_all_folds(
        X_df, outer_cv, panel_dir=panel_dir, shap_dir=shap_dir,
        target="A", feature_names=feature_names,
        top_k=top_k, max_points=5000, seed=seed,
        outpath=plots_dir / f"beeswarm_{tslugA}.png"
    )

    _log(f"Plotting beeswarm ({target_labels['B']}) ...")
    plot_shap_beeswarm_all_folds(
        X_df, outer_cv, panel_dir=panel_dir, shap_dir=shap_dir,
        target="B", feature_names=feature_names,
        top_k=top_k, max_points=5000, seed=seed,
        outpath=plots_dir / f"beeswarm_{tslugB}.png"
    )

    _log(f"Plotting bar ±SD (top-{top_k}, {target_labels['A']}) ...")
    plot_shap_bar_crossfold(
        shap_dir / "shap_meanabs_crossfold.csv",
        target="A", top_k=top_k,
        gene_map_csv=gene_map_csv, disambiguate_symbols=True,
        outpath=plots_dir / f"bar_top{top_k}_{tslugA}.png"
    )

    _log(f"Plotting bar ±SD (top-{top_k}, {target_labels['B']}) ...")
    plot_shap_bar_crossfold(
        shap_dir / "shap_meanabs_crossfold.csv",
        target="B", top_k=top_k,
        gene_map_csv=gene_map_csv, disambiguate_symbols=True,
        outpath=plots_dir / f"bar_top{top_k}_{tslugB}.png"
    )

    _log(f"Plotting stability (top-{top_k}, {target_labels['A']}) ...")
    plot_shap_stability_dot(
        shap_dir / "shap_meanabs_crossfold.csv",
        target="A", top_k=top_k,
        gene_map_csv=gene_map_csv, disambiguate_symbols=True,
        outpath=plots_dir / f"stability_top{top_k}_{tslugA}.png"
    )

    _log(f"Plotting stability (top-{top_k}, {target_labels['B']}) ...")
    plot_shap_stability_dot(
        shap_dir / "shap_meanabs_crossfold.csv",
        target="B", top_k=top_k,
        gene_map_csv=gene_map_csv, disambiguate_symbols=True,
        outpath=plots_dir / f"stability_top{top_k}_{tslugB}.png"
    )

    _log(f"Plotting biplot A vs B (log_scale={do_biplot_log}) ...")
    plot_shap_biplot_AB(
        shap_cross_csv=shap_dir / "shap_meanabs_crossfold.csv",
        target_labels=target_labels,
        gene_map_csv=gene_map_csv,
        log_scale=do_biplot_log,
        min_value=biplot_min_value,
        pad_frac=biplot_pad_frac,
        cap_by_percentile=biplot_cap_percentile,
        annotate_top_k=biplot_annotate_top_k,
        outpath=plots_dir / ("biplot_A_vs_B_log.png" if do_biplot_log else "biplot_A_vs_B.png")
    )

    _log("Done.")
    return cross


In [3]:
OUT_XGB_PANEL = Path("../data/ml_output/mt_en_subset_panels_thr070/mt_en_union_thr70/")
outer_cv = KFold(n_splits=10, shuffle=True, random_state=SEED)

In [ ]:
cross = run_shap_for_panel(
    X_df=X_union,
    y_mat=Y.values.astype(np.float32),
    outer_cv=outer_cv,
    panel_dir=OUT_XGB_PANEL,
    target_labels={'A':'Propensity','B':'Flux Intensity'},
    gene_map_csv="../data/features_label/gene_annotation/gene_info.csv",
    top_k=20,
    background_subsample=400,   
    nsamples_kernel=512,
    n_jobs_outer=30,
    do_biplot_log=True,
    biplot_min_value=1e-6
)

In [39]:
OUT_EN_PANEL = Path("../data/ml_output/mt_en_subset_panels_thr070/mt_en_union_thr70/")
outer_cv = KFold(n_splits=10, shuffle=True, random_state=SEED)

In [ ]:
cross = run_shap_for_panel(
    X_df=X_union,
    y_mat=Y.values.astype(np.float32),
    outer_cv=outer_cv,
    panel_dir=OUT_EN_PANEL,
    target_labels={'A':'Propensity','B':'Flux Intensity'},
    gene_map_csv="../data/features_label/gene_annotation/gene_info.csv",
    top_k=20,
    background_subsample=400,   
    nsamples_kernel=512,
    n_jobs_outer=30,
    do_biplot_log=True,
    biplot_min_value=1e-6
)

[15:38:11] Loading gene map: ../data/features_label/gene_annotation/gene_info.csv
[15:38:11] SHAP (XGB/Kernel) — bg=400, nsamples=512, folds=10
[15:42:55] Saved cross-fold summary -> ..\data\ml_output\mt_en_subset_panels_thr070\mt_en_union_thr70\feature_importance\shap\shap_meanabs_crossfold.csv (rows=872)
[15:42:55] Plotting beeswarm (Propensity) ...
[15:42:56] Plotting beeswarm (Flux Intensity) ...
[15:42:57] Plotting bar ±SD (top-20, Propensity) ...
[15:42:57] Plotting bar ±SD (top-20, Flux Intensity) ...
[15:42:58] Plotting stability (top-20, Propensity) ...
[15:42:58] Plotting stability (top-20, Flux Intensity) ...
[15:42:58] Plotting biplot A vs B (log_scale=True) ...
[15:42:59] Done.


<Figure size 640x480 with 0 Axes>